# Sesión 5 — Buenas prácticas de data lake y particionado físico

Esta sesión se sitúa justo después de S4: ya sabemos leer Parquet con
Spark, construir DataFrames y ejecutar transformaciones tanto en local como
sobre YARN. Toca dar un paso que no es de motor sino de organización: un
data lake no es solamente un directorio lleno de ficheros Parquet. También
necesita convenciones de nombres, contratos documentados, comprobaciones de
calidad y una estrategia de mantenimiento, para que el mismo conjunto de
datos pueda reutilizarse, entenderse y consultarse eficientemente sin
convertirlo todavía en una tabla gestionada por un catálogo.

La pregunta central de esta sesión es:

> ¿Cómo organizamos los ficheros para que sean reutilizables, comprensibles
> y eficientes sin convertir todavía el data lake en una tabla gestionada?

Para responderla mantendremos el mismo TPC-DS SF3 de las sesiones
anteriores y compararemos tres versiones del mismo dato: la fuente *raw*
que ya conoces de S2, una versión refinada *silver* que construiremos con
Spark, y un agregado de negocio *gold* derivado de esa versión refinada.

La figura sitúa la sesión en la arquitectura física del laboratorio: cada círculo es un contenedor Docker, y los puntos de color de su interior son los servicios que ejecuta; a la derecha están los servicios, con el color de la sesión que los introduce y una flecha hacia el clúster. En color está lo que ya existe al empezar la sesión y sombreado, lo que llegará en sesiones posteriores; resaltado, lo que esta sesión crea, modifica o usa. S5 usa Spark para escribir en el data lake las zonas `silver` y `gold`, que hasta ahora estaban vacías.

<p align="center" style="text-align: center;"><img src="https://dsevilla.github.io/tcdm-public/figs/arquitectura-s5.svg" alt="Arquitectura del laboratorio al empezar la sesión 5: Spark escribe las zonas silver y gold del data lake" width="1100" style="display: block; margin: 0 auto; max-width: 100%; height: auto;"></p>

Al terminar S5 seguimos teniendo un data lake basado en ficheros: que los
directorios sigan el patrón `columna=valor` no crea por sí mismo una tabla
permanente ni proporciona snapshots, transacciones o evolución de esquema.
Eso llegará con el Hive Metastore en S6 y con Iceberg en S7. Lo que sí
habremos hecho es aplicar, de forma observable, la arquitectura *medallion*
que ya se mencionó al crear el data lake en S1 y al explorar la zona `raw`
en S2: datos detallados y reutilizables en `silver`, y un agregado orientado
a una pregunta de negocio en `gold`.

> **Paquetes de Python de esta sesión.** S5 usa `pyspark` para transformar
> y escribir las zonas `silver` y `gold`; no introduce un paquete nuevo
> respecto a S4. Todos los notebooks de S2-S8 instalan la lista común publicada
> como `entorno/requirements.txt`, que reúne los paquetes usados a lo largo
> del curso. La imagen de `namenode` ya trae esa lista en `/opt/tcdm/venv`, el
> intérprete del kernel; lo normal es que `%pip` confirme que están.

## Actualizar el material antes de empezar

Antes de empezar, desde la raíz de la distribución y en una terminal de tu
equipo:

```bash
make -C entorno actualizar
```

La orden trae la última versión publicada del material. No pisa tu trabajo: si
has modificado un fichero que también cambia en la versión nueva, se aparta
como copia `<fichero>.mio-<fecha>-<hora>` y el original pasa a ser el
publicado. Lo que la versión nueva no cambia se queda como está.


### Consultar el esquema de TPC-DS

Como en S2, `%esquema` muestra debajo de la celda en la que lo escribas la
descripción de una tabla o de una columna de TPC-DS, tomada del apéndice de
`s2.ipynb`:

| Celda | Qué muestra |
| --- | --- |
| `%esquema` | Las 18 tablas, con su papel y su descripción |
| `%esquema web_sales` | La tabla completa: su descripción, su diagrama entidad-relación si es una tabla de hechos, y todas sus columnas |
| `%esquema web_sales date` | Las columnas de `web_sales` cuyo nombre contiene `date` |
| `%esquema ws_sold_timestamp` | Esa columna y ninguna otra |

Una celda `%%esquema` admite una tabla, columna o filtro por línea. Lo
encontrarás la primera vez que la sesión usa cada tabla; añade una celda de
éstas donde te haga falta y bórrala después.

`%schema` y `%%schema` son los mismos comandos con su nombre en inglés:
usa el que prefieras.

La celda siguiente lo prepara: ejecútala una vez por sesión de kernel.
Descarga `entorno/python_utils/tcdm_esquema.py` del repositorio público
`dsevilla/tcdm-public` (rama `26-27`), porque el kernel se ejecuta en
`namenode`, que no ve tu copia de la distribución. Si el fichero ya está en
el directorio de trabajo del kernel, se conserva; bórralo para traer una
versión nueva.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen

SCHEMA_MODULE: Path = Path("tcdm_esquema.py")
schema_module_url: str = (
    "https://raw.githubusercontent.com/dsevilla/tcdm-public/26-27/"
    "entorno/python_utils/tcdm_esquema.py"
)

if not SCHEMA_MODULE.exists():
    request: Request = Request(schema_module_url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=30) as response:
        SCHEMA_MODULE.write_bytes(response.read())

%load_ext tcdm_esquema

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:linear-gradient(135deg,#0c304d,#174f7a 62%,#a54467);color:#ffffff;border:1px solid transparent;display:flex;flex-direction:column;justify-content:center;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="margin:0 auto;max-width:44ch;text-align:left"><h1 align="center" style="color:#ffffff;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;text-align:center;margin:0 0 22px">Sesión 5: buenas prácticas de data lake y particionado físico</h1><h2 align="center" style="color:#dceaf4;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:30px;text-align:center;margin:18px 0 8px">Cómo vamos a recorrer la sesión</h2><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>El mismo TPC-DS SF3 en tres versiones: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code></li><li style='margin:10px 0;list-style-position:outside'>Contratos, calidad y mantenimiento: un data lake no es un directorio lleno de ficheros</li><li style='margin:10px 0;list-style-position:outside'>Spark escribe una tabla <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> particionada y un producto <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code></li><li style='margin:10px 0;list-style-position:outside'>La evidencia de que particionar ayuda se lee en el plan y en los ficheros leídos; el reloj sólo la acompaña</li></ul></div><hr color="#174f7a" size="8" noshade style="display:none"></div>

## Objetivos de la sesión

Al terminar esta sesión deberías poder:

- explicar la arquitectura medallion (`raw`/`silver`/`gold`) y qué papel
  cumple cada zona en un data lake basado en ficheros;
- escribir el contrato de una salida de datos: rutas, columnas y sus
  tipos (declarados en código como un esquema de Spark), claves de unión,
  significado de cada métrica, formato y regla de regeneración;
- distinguir el particionado físico por directorios `columna=valor` de
  otras nociones que comparten la palabra "partición": las particiones de
  ejecución de Spark, los bloques HDFS, los *splits* de un motor SQL y las
  transformaciones de partición de una tabla Iceberg;
- justificar la elección de columnas de partición a partir de los patrones
  de consulta y de la cardinalidad de los datos, no por costumbre;
- construir con Spark una tabla `silver` particionada por año y mes a
  partir de `web_sales`, conservando una salida de control sin
  particionar para comparar;
- validar, con comprobaciones ejecutables (no sólo visuales), que una
  salida cumple su esquema declarado, que una transformación no pierde
  más filas de las esperadas, que las columnas derivadas no aparecen
  inesperadamente nulas y que no faltan las claves que identifican cada
  línea;
- agregar desde `silver` un producto `gold` orientado a una pregunta de
  negocio y distinguir la métrica de pedidos de la métrica de líneas de
  venta;
- observar el descubrimiento automático de particiones al releer una tabla
  particionada por su ruta raíz, y comparar una lectura completa con una
  lectura filtrada mirando el plan físico y los ficheros efectivamente
  leídos, y usar el tiempo de ejecución sólo como indicio complementario;
- comprobar con un ejercicio que una consulta por otra clave (el cliente)
  que no se beneficia de la partición por año y mes sí mejora con una
  segunda tabla de laboratorio ordenada por esa clave;
- describir qué límites sigue teniendo una colección de Parquet
  particionada sin catálogo, como preparación para S6.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #b3701a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#b3701a" size="8" noshade style="display:none"><div style="color:#b3701a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#b3701a" style="color:inherit"><b>Evaluación de la sesión</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Cómo se evalúa esta sesión</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Reunión individual de unos 5 minutos con el profesor</li><li style='margin:10px 0;list-style-position:outside'>Se muestra en vivo, en tu ordenador: el árbol de directorios <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_year=/sold_month=</code> de la tabla <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code>, los <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>assert</code> de filas y claves superados y la comparación de ficheros leídos entre la consulta completa y la filtrada</li><li style='margin:10px 0;list-style-position:outside'>Se entrega también una memoria breve (una o dos páginas)</li><li style='margin:10px 0;list-style-position:outside'>No hace falta memorizar órdenes: sí explicar qué garantiza cada zona del data lake y qué demuestra cada comprobación</li></ul><hr color="#b3701a" size="2" noshade style="display:none"></div>

## Antes de empezar

> **Dónde se ejecuta este notebook.** Igual que en S2, S3 y S4, Jupyter se
> ejecuta directamente dentro del contenedor `namenode`, como `luser`, con
> acceso de red directo a HDFS y YARN. Las celdas ejecutables hablan
> directamente con el clúster. Las órdenes que necesitan el Docker del
> host —arrancar o detener contenedores, `make`— se muestran como texto
> para ejecutarlas en una terminal de tu equipo, nunca como celdas de este
> notebook.

Esta sesión asume que el clúster de S1 ya está en marcha y que S2 dejó
materializado TPC-DS SF3 en `/datalake/raw/tpcds` (S4 sólo lo leyó). S5 no
vuelve a generar esos datos ni los modifica: se limita a leerlos. Si el clúster no
está arriba, desde una terminal de tu equipo situada en la raíz de la
distribución de sesiones:

<p align="center" style="text-align: center;"><img src="https://dsevilla.github.io/tcdm-public/figs/anim-host-hadoop-up.gif" alt="Animación: la orden make -C entorno hadoop-up se teclea en una terminal de tu equipo (host)" width="832" style="display: block; margin: 0 auto; max-width: 100%; height: auto;"></p>

```bash
cd ~/tcdm-public
make -C entorno hadoop-up
```

Este notebook se ejecuta en el Jupyter de `namenode`, que no arranca solo.
Como en S2, arráncalo desde **otra terminal de tu equipo** y déjala abierta
mientras trabajes:

<p align="center" style="text-align: center;"><img src="https://dsevilla.github.io/tcdm-public/figs/anim-namenode-jupyter.gif" alt="Animación: docker exec -it namenode bash se teclea en tu equipo y abre una terminal dentro del contenedor namenode, donde se ejecutan su - luser y jupyter lab" width="832" style="display: block; margin: 0 auto; max-width: 100%; height: auto;"></p>

```bash
# en tu equipo: entra en namenode
docker exec -it namenode bash
# ya dentro de namenode, como luser: Jupyter con el token fijo «tcdm»
su - luser
jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --IdentityProvider.token=tcdm
```

La primera orden es la única que se ejecuta en tu equipo; las otras dos, ya
dentro del contenedor, arrancan Jupyter con el Python del curso
(`make -C entorno jupyter` hace lo mismo en una sola orden). En Visual
Studio Code, conecta este notebook con **Select Kernel → Select Another
Kernel → Existing Jupyter Server**, usando la dirección
`http://127.0.0.1:8888/lab?token=tcdm` y eligiendo el kernel **Python 3
(ipykernel)**. El token es siempre `tcdm`.

Si algo no funciona —el notebook no conecta, una celda `!hdfs` no
encuentra la orden, un servicio no responde—, consulta «Solución de
problemas» en `entorno/README.md`.

Antes de instalar nada, comprueba que la fuente sigue donde S2 la dejó y S4
la utilizó:

In [ ]:
!hdfs dfs -chmod -R o+rX /datalake /warehouse
!hdfs dfs -ls -h -R /datalake/raw/tpcds

Si el directorio no existe, revisa S2 antes de continuar: esta sesión no
regenera TPC-DS. Tampoco vamos a tocar nada bajo esa ruta: todas las
escrituras de este notebook van a `/datalake/silver`, `/datalake/gold` o a
la ruta de trabajo `/user/luser/s5`.

## Instalar PySpark en este notebook

Como en S4, el kernel instala PySpark y construye una única
`SparkSession` sobre YARN, que se reutiliza en todo el notebook.

La distribución publica una sola lista común en `entorno/requirements.txt`,
que reúne las dependencias usadas de S2 a S8; S5 no necesita una lista
distinta ni añade una dependencia a la lista común. La celda siguiente la
instala con `%pip` directamente desde su URL HTTPS: `pip` admite una URL en
`-r`, así que no hace falta escribir una copia local del fichero. La imagen
de `namenode` ya trae estos paquetes en `/opt/tcdm/venv`, así que lo normal
es que `%pip` confirme que están.

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/dsevilla/tcdm-public/26-27/entorno/requirements.txt

In [ ]:
import sys

import pyspark

print(f"PySpark {pyspark.__version__} con Python {sys.version.split()[0]}.")

## Crear una SparkSession sobre YARN

Como en la parte distribuida de S4, el driver vive en `namenode` y YARN
asigna los ejecutores a los NodeManagers. La celda siguiente cuenta los que
están `RUNNING` y solicita, por defecto, un ejecutor por cada uno; YARN decide
su colocación, por lo que no se garantiza uno en cada DataNode. Para pedir
otra cantidad, fija antes `%env TCDM_SPARK_EXECUTORS=5` en una celda del
notebook, como en S4. Cierra cualquier otra sesión Spark antes de continuar
para que no retenga recursos ni ocupe el puerto 4040.

Cada ejecutor pide un núcleo, 512 MiB de memoria y 256 MiB adicionales;
el ApplicationMaster pide 512 MiB en total. Cada NodeManager anuncia
2560 MiB. La sesión y sus contenedores se liberan con `spark.stop()`.
Si faltan NodeManagers, comprueba el clúster desde una terminal de tu equipo
o en el ResourceManager (`http://localhost:8088`).

In [ ]:
import os
import subprocess

assert os.environ.get("HADOOP_CONF_DIR"), (
    "El kernel no hereda HADOOP_CONF_DIR: arranca Jupyter como luser dentro "
    "de namenode, como se explica en «Antes de empezar»."
)
node_result: subprocess.CompletedProcess[str] = subprocess.run(
    ["yarn", "node", "-list", "-all"], capture_output=True, text=True, check=True
)
running_nodemanagers: int = sum(
    1
    for line in node_result.stdout.splitlines()
    if len(line.split()) >= 2 and line.split()[1] == "RUNNING"
)
print(f"NodeManagers en estado RUNNING: {running_nodemanagers}")
assert running_nodemanagers >= 2, (
    "Se esperan al menos dos NodeManagers RUNNING. Comprueba los DataNodes "
    "con `make -C entorno status` en una terminal de tu equipo o en "
    "http://localhost:8088; si acabas de arrancar el clúster, espera y repite."
)
requested_executors: int = int(os.environ.get("TCDM_SPARK_EXECUTORS", running_nodemanagers))
assert requested_executors > 0, "TCDM_SPARK_EXECUTORS debe ser un entero positivo."
print(f"Solicitando {requested_executors} ejecutores Spark a YARN.")

In [ ]:
from pyspark import SparkContext
from pyspark.sql import SparkSession

spark: SparkSession = (
    SparkSession.builder.appName("TCDM_S5_YARN")
    .master("yarn")
    .config("spark.executor.instances", str(requested_executors))
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.memoryOverhead", "256m")
    .config("spark.yarn.am.cores", "1")
    .config("spark.yarn.am.memory", "256m")
    .config("spark.yarn.am.memoryOverhead", "256m")
    .config("spark.scheduler.minRegisteredResourcesRatio", "1.0")
    .config("spark.scheduler.maxRegisteredResourcesWaitingTime", "60s")
    .config("spark.pyspark.python", "python3")
    .config("spark.yarn.isPython", "true")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
sc: SparkContext = spark.sparkContext
assert sc.master == "yarn", (
    "Ya hay una sesión Spark con otro master. Ciérrala con spark.stop() "
    "y vuelve a ejecutar esta celda."
)

print(f"Spark version: {spark.version} | Master: {sc.master} | App: {sc.appName}")

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Las convenciones antes del código</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Zona fuente inmutable: por qué <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/datalake/raw/tpcds</code> no se toca</li><li style='margin:10px 0;list-style-position:outside'>Contratos: ruta, columnas, claves, métricas, formato y regeneración</li><li style='margin:10px 0;list-style-position:outside'>Esquema y calidad: qué se comprueba antes de dar una salida por buena</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Buenas prácticas de data lake

Un data lake de ficheros no deja de ser un data lake por no tener un
catálogo permanente: puede ser correcto, reproducible y eficiente si se
siguen algunas convenciones básicas. Esta sección describe esas
convenciones antes de aplicarlas en código.

### Conservar una zona fuente

Los Parquet de `/datalake/raw/tpcds` representan la fuente reproducible del
curso. Ninguna transformación de esta sesión los sobrescribe. Conservar
una fuente inmutable permite repetir una sesión, comparar lectores y
motores, reconstruir una tabla refinada si cambia el código de
transformación, comprobar que una transformación no altera los datos
originales, y mantener una referencia estable frente a la que comprobar
cualquier resultado derivado.

La organización lógica del data lake del curso puede describirse mediante
tres zonas. Las etiquetas *raw*, *silver* y *gold* son una ayuda de diseño,
no una garantía de calidad por sí mismas: cada conjunto debe documentar su
origen, su transformación, su esquema y quién es responsable de
mantenerlo.

| Zona | Función | Aplicación en el curso |
| --- | --- | --- |
| Fuente o *raw* | Datos originales, conservados e inmutables | `/datalake/raw/tpcds` |
| Refinada o *silver* | Datos limpiados, enriquecidos y reorganizados | `/datalake/silver/tpcds` |
| Curada o *gold* | Productos preparados para una pregunta o consumidor concreto | `/datalake/gold/tpcds` |

Esta terminología sigue la [arquitectura medallion documentada por
Databricks](https://docs.databricks.com/aws/en/lakehouse/medallion): los
datos ganan estructura y calidad a medida que avanzan desde la ingestión
hasta los productos destinados al consumo. El segundo nivel de cada ruta
identifica la fuente y permite incorporar otras colecciones sin mezclarlas:
por ejemplo, `/datalake/raw/weather` o `/datalake/raw/prices` convivirían
con `/datalake/raw/tpcds` sin interferir entre sí. Una transformación que
algún día cruzara esas fuentes con TPC-DS escribiría un conjunto nuevo en
`silver` o un producto específico en `gold`, dejando intactas las entradas
`raw`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Un compañero propone sobrescribir <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/datalake/raw/tpcds</code> con los datos ya limpios para ahorrar espacio: ¿qué dejarías de poder hacer?</li><li style='margin:10px 0;list-style-position:outside'>Si <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> se estropea, ¿de dónde se reconstruye? ¿Y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code>?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Por qué las capas generadas necesitan un contrato</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code> llega de fuera y no se toca; <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code> las genera nuestro código a partir de otras tablas</li><li style='margin:10px 0;list-style-position:outside'>Los ficheros Parquet de una tabla generada no dicen de dónde vienen, qué se descartó ni cómo se calculó cada columna</li><li style='margin:10px 0;list-style-position:outside'>Cada capa se apoya en la anterior: un cambio sin documentar en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> llega a <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code> sin que nadie lo vea</li><li style='margin:10px 0;list-style-position:outside'>El contrato de cada tabla generada fija su origen y linaje, su esquema, su regeneración y su responsable</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

### Por qué `silver` y `gold` necesitan un contrato

La zona `raw` tiene un origen externo: alguien ingirió esos ficheros y
el curso se compromete a no tocarlos. `silver` y `gold` son distintas:
no las trae nadie de fuera, **las genera nuestro código** a partir de
otras tablas del data lake. Esa diferencia obliga a dejar por escrito
cosas que en `raw` no hacía falta explicar. Una tabla generada no se
explica por sí sola: viendo sus ficheros Parquet no se sabe de qué
tabla de partida salen, qué filas y columnas se descartaron por el
camino, cómo se calculó cada columna derivada, ni cómo volver a
producirla si se estropea. Además, las capas dependen unas de otras
(`gold` se construye sobre `silver`, y `silver` sobre `raw`), de modo
que un cambio o un error no documentado en una capa se propaga a las
siguientes sin que quien las consume pueda verlo.

Por eso, en una arquitectura con capas generadas, cada tabla de `silver`
y de `gold` lleva un **contrato**: la descripción escrita y comprobable
de qué es esa tabla y cómo se obtiene. Un contrato cubre, al menos:

- **Origen y linaje (*lineage*)**: de qué tablas y columnas parte y con
  qué transformación. Permite reconstruir la tabla y saber qué
  productos posteriores se ven afectados si cambia una de sus fuentes.
- **Esquema**: columnas, orden, tipos, claves y valores que pueden o no
  ser nulos. Es lo que el consumidor necesita para usar la tabla sin
  leer el código que la generó.
- **Significado**: qué representa cada columna o métrica, y qué no debe
  confundirse con ella.
- **Forma de almacenamiento**: ruta, formato, compresión y
  particionado.
- **Regeneración y responsable**: cómo se vuelve a producir y quién
  responde de ella y de sus cambios.

El contrato sirve a dos públicos. Quien consume la tabla sabe qué puede
asumir sobre ella; quien la mantiene sabe qué debe seguir cumpliéndose
cuando cambie la transformación, y puede convertir esa descripción en
comprobaciones automáticas, que es lo que haremos con el esquema más abajo, en «Los esquemas de
`silver` y `gold`».

En esta sesión generamos sólo dos tablas, y por eso sólo escribimos dos
contratos: los de `/datalake/silver/tpcds/web_sales` y
`/datalake/gold/tpcds/web_sales_by_year`. Son un **ejemplo**: en un data
lake real habría que especificar de este modo todas y cada una de las
tablas de `silver` y de `gold`, y no se debería publicar ninguna sin su
contrato.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Contratos y nombres: el ejemplo de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales</code></h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Cada tabla generada necesita su contrato; aquí lo escribimos para las dos que genera esta sesión, como ejemplo</li><li style='margin:10px 0;list-style-position:outside'><strong>Origen:</strong> <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> sale de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales</code> de TPC-DS SF3 en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code>; <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code> sale de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code></li><li style='margin:10px 0;list-style-position:outside'><strong>Columnas de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code>:</strong> 11 de las 33 de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales</code> más <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_sold_date</code>; las demás se quedan en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code></li><li style='margin:10px 0;list-style-position:outside'><strong>Derivadas:</strong> <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_sold_date</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_year</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_month</code> se calculan a partir de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_sold_timestamp</code></li><li style='margin:10px 0;list-style-position:outside'><strong>Claves:</strong> <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_order_number</code> + <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_item_sk</code> identifican la línea; las claves de unión con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>item</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>customer</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>customer_address</code> pueden venir nulas</li><li style='margin:10px 0;list-style-position:outside'><strong>Métricas de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code>:</strong> pedidos distintos, líneas de venta y beneficio neto, una fila por año (2003 sólo cubre dos días)</li><li style='margin:10px 0;list-style-position:outside'><strong>Esquema:</strong> columnas y tipos de cada salida, declarados como esquema de Spark en la sección siguiente</li><li style='margin:10px 0;list-style-position:outside'><strong>Formato:</strong> Parquet con Snappy, el códec por defecto de Spark (<code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code> usa ZSTD)</li><li style='margin:10px 0;list-style-position:outside'><strong>Regeneración y responsable:</strong> este notebook, en modo <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>overwrite</code></li><li style='margin:10px 0;list-style-position:outside'><strong><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>_SUCCESS</code>:</strong> marcador de generación, no un dato</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

### Contratos y nombres

Cada tabla generada debe tener una ruta estable y documentada. Como
ejemplo, escribimos aquí el contrato de las dos tablas que genera esta
sesión: `/datalake/silver/tpcds/web_sales` y
`/datalake/gold/tpcds/web_sales_by_year`. Lo documentamos en este
mismo notebook, en lugar de dejarlo sólo en la cabeza de quien escribió
el código, y para cada una de las dos tablas indica:

- **Origen:** `silver/tpcds/web_sales` se genera a partir de la tabla
  `web_sales` de TPC-DS SF3 que está en `raw`; la tabla
  `gold/tpcds/web_sales_by_year` se genera a partir de `silver`, no de
  `raw`.
- **Columnas de `silver`:** conserva 11 de las 33 columnas de
  `web_sales`, las que usan las consultas que esperamos sobre la tabla
  (se enumeran en la sección de transformación) —la línea (pedido y
  artículo), quién compra y dónde factura, la marca de tiempo, la
  cantidad y los importes de venta, descuento, pago y beneficio—, y
  añade `ws_sold_date` (derivada) y las particiones `sold_year` y
  `sold_month`, en este orden y con los tipos que fija el esquema. Las
  otras 22 (envío, perfil demográfico, página y sitio web, almacén,
  promoción, costes mayoristas, impuestos y cupones) se quedan sólo en
  `raw`: quien las necesite las lee de allí.
- **Columnas derivadas:** `ws_sold_date`, `sold_year` y `sold_month` se
  calculan a partir de `ws_sold_timestamp`, con `to_date`, `year` y
  `month`; las líneas sin fecha no entran en `silver`.
- **Claves:** `ws_order_number` y `ws_item_sk` identifican juntas cada
  línea de venta y no pueden faltar; `ws_item_sk`, `ws_bill_customer_sk`
  y `ws_bill_addr_sk` son las claves de unión con `item`, `customer` y
  `customer_address`. Las dos últimas pueden venir nulas desde la
  fuente —TPC-DS deja a propósito algunas líneas sin cliente o sin
  domicilio de facturación, como viste en S2 y S4—, y `silver` las
  conserva tal cual: quien la cruce con `customer` o `customer_address`
  debe saber que esas líneas no encontrarán pareja.
- **Métricas de `gold`:** pedidos distintos, líneas de venta y beneficio
  neto, sin confundir unas con otras.
- **Ubicación y columnas de `gold`:** una fila por año en
  `/datalake/gold/tpcds/web_sales_by_year`, con la columna `sold_year` y
  esas tres métricas. La fila de 2003 sólo cubre el 1 y el 2 de enero,
  porque ahí terminan los datos de TPC-DS, y no debe compararse con los
  años completos.
- **Formato y compresión:** Parquet comprimido con Snappy, el códec por
  defecto de Spark (`spark.sql.parquet.compression.codec`). `raw`, que
  escribió DuckDB en S2, usa ZSTD: las zonas no tienen por qué compartir
  códec, pero el contrato de cada tabla debe decir cuál usa.
- **Particionado:** `silver` se particiona por `sold_year` y
  `sold_month`, en ese orden: se escribe en directorios
  `sold_year=.../sold_month=...`. Qué es el particionado y por qué se eligen
  esas dos columnas se explica más abajo, en «Por qué particionar un data
  lake de ficheros» y en «Diseño de las particiones».
- **Regeneración:** `silver` y `gold` se reconstruyen volviendo a
  ejecutar las celdas de la sección "Ejercicio de transformación con
  Spark" de este notebook, que escriben siempre en modo `overwrite`
  (primero `silver`, y `gold` a partir de `silver`, no de `raw`).
- **Responsable:** este notebook de S5 es el único que escribe `silver`
  y `gold`. Quien cambie su transformación cambia a la vez este
  contrato, los esquemas declarados en la sección siguiente y las
  sesiones que dependen de ellos (las tablas de S6 y el esquema de S8).
- **Ficheros auxiliares:** `_SUCCESS` es un marcador de generación, no
  un fragmento Parquet, y no debe incluirse en los patrones de lectura.

Las columnas y sus tipos no se quedan en esta lista: la sección siguiente
los escribe como esquemas de Spark, que el notebook usará para comprobar
lo que escribe y para leer lo que ha escrito. El resto del contrato
(origen, significado, regeneración y responsable) queda aquí, en prosa,
y es lo que debería escribirse igualmente para cualquier otra tabla
generada.

### Los esquemas de `silver` y `gold`

Un **esquema** de Spark es la lista ordenada de columnas de un
DataFrame, cada una con su nombre, su tipo y si admite nulos. Es lo que
imprime `printSchema()`, y también un objeto de Python: un `StructType`
que contiene un `StructField` por columna. S4 ya construyó uno en el
trabajo de YARN para dar forma al informe de particiones; aquí lo
usamos para lo contrario, para escribir antes de transformar nada qué
forma deben tener las dos salidas de la sesión.

Los tipos de las columnas que `silver` conserva de `web_sales` son los
del catálogo de S2 (`%esquema web_sales`), es decir, los que escribió
DuckDB. En Spark se llaman de otra forma:

| Tipo en S2 | Tipo de Spark | Columnas |
| --- | --- | --- |
| `bigint` | `LongType()` | claves, `ws_order_number` y `ws_quantity` |
| `decimal(7,2)` | `DecimalType(7, 2)` | importes |
| `timestamp` (sin zona) | `TimestampNTZType()` | `ws_sold_timestamp` |

Las columnas que calcula esta sesión tienen el tipo que devuelve la
función que las calcula: `to_date` da un `DateType()`; `year` y `month`,
un `IntegerType()`; `count` y `countDistinct`, un `LongType()`; y `sum`
sobre un `decimal(7,2)` da un `decimal(17,2)`, diez dígitos más para que
la suma de muchas filas no se desborde.

Todas las columnas se declaran con `nullable=True`, aunque algunas no
deban tener nulos nunca. Spark trata como nullable toda columna que lee
de un fichero Parquet, diga lo que diga el fichero o el esquema que se
le pase, así que un `nullable=False` en el contrato no se cumpliría al
releer la salida. Que una columna no tenga nulos lo comprueban los
`assert` de la sección de transformación, no el esquema.

In [ ]:
from pyspark.sql.types import (
    DateType,
    DecimalType,
    IntegerType,
    LongType,
    StructField,
    StructType,
    TimestampNTZType,
)

# Contrato de /datalake/silver/tpcds/web_sales: columnas, orden y tipos.
# sold_year y sold_month son las columnas de partición: van al final.
web_sales_silver_schema: StructType = StructType(
    [
        StructField("ws_order_number", LongType(), nullable=True),
        StructField("ws_item_sk", LongType(), nullable=True),
        StructField("ws_bill_customer_sk", LongType(), nullable=True),
        StructField("ws_bill_addr_sk", LongType(), nullable=True),
        StructField("ws_sold_timestamp", TimestampNTZType(), nullable=True),
        StructField("ws_sold_date", DateType(), nullable=True),
        StructField("ws_quantity", LongType(), nullable=True),
        StructField("ws_list_price", DecimalType(7, 2), nullable=True),
        StructField("ws_ext_discount_amt", DecimalType(7, 2), nullable=True),
        StructField("ws_ext_sales_price", DecimalType(7, 2), nullable=True),
        StructField("ws_net_paid", DecimalType(7, 2), nullable=True),
        StructField("ws_net_profit", DecimalType(7, 2), nullable=True),
        StructField("sold_year", IntegerType(), nullable=True),
        StructField("sold_month", IntegerType(), nullable=True),
    ]
)

# Contrato de /datalake/gold/tpcds/web_sales_by_year: una fila por año.
web_sales_by_year_schema: StructType = StructType(
    [
        StructField("sold_year", IntegerType(), nullable=True),
        StructField("pedidos", LongType(), nullable=True),
        StructField("lineas_de_venta", LongType(), nullable=True),
        StructField("beneficio_neto", DecimalType(17, 2), nullable=True),
    ]
)

for field in web_sales_silver_schema.fields:
    print(f"silver  {field.name:<20} {field.dataType.simpleString()}")
for field in web_sales_by_year_schema.fields:
    print(f"gold    {field.name:<20} {field.dataType.simpleString()}")

La salida usa el nombre de cada tipo en SQL (`bigint`, `int`,
`decimal(7,2)`, `timestamp_ntz`): son los mismos tipos que el catálogo de
S2 y que la definición de tabla que escribirá S6 al registrar estas dos
rutas en el metastore. El esquema se usa tres veces en el notebook:
antes de escribir cada salida, para comprobar que el DataFrame que se va
a escribir lo cumple; al releer cada salida, para comprobar que lo que
ha quedado en HDFS es lo declarado; y al leer `silver` para construir
`gold`, como lo leería cualquier consumidor de esa tabla.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Qué comprobar antes de dar una salida por buena</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>El esquema es el declarado, en nombres, orden y tipos</li><li style='margin:10px 0;list-style-position:outside'>Las claves que identifican la fila no faltan y los rangos tienen sentido (<code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_month</code> entre 1 y 12)</li><li style='margin:10px 0;list-style-position:outside'>El número de filas cuadra con la fuente: sólo se pierden las filas que el contrato descarta</li><li style='margin:10px 0;list-style-position:outside'>La métrica distingue líneas de venta de pedidos distintos</li><li style='margin:10px 0;list-style-position:outside'>Más abajo serán <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>assert</code> ejecutables, no comentarios</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

### Esquema y calidad

Antes de dar por buena una salida —y desde luego antes de que otra
sesión la registre como tabla— conviene comprobar al menos:

- que el esquema es el declarado en el contrato, en nombres, orden y
  tipos;
- que las claves que identifican cada fila no aparecen nulas, y cuántas
  claves de unión vienen vacías desde la fuente;
- que el número de filas cuadra con la fuente: sólo se pierden las que
  el contrato descarta;
- que los rangos de fechas y claves tienen sentido (por ejemplo, que
  `sold_month` esté siempre entre 1 y 12);
- si la salida sale de cruzar tablas, que el `JOIN` no pierde ni
  multiplica filas por error, como comprobaste en S4 con los recuentos de
  `INNER`, `LEFT OUTER` y `LEFT ANTI` (las salidas de esta sesión no
  cruzan tablas);
- que la métrica de negocio distingue líneas de venta de pedidos
  distintos;
- que los ficheros se pueden leer con más de un lector cuando sea
  relevante (ya lo comprobamos en S2 con PyArrow, Polars y DuckDB sobre la
  misma fuente).

Estas comprobaciones no sustituyen a un sistema de calidad de producción,
pero enseñan que escribir ficheros Parquet correctamente no basta, por sí
solo, para tener datos fiables. Más abajo, en la sección de transformación,
convertiremos varias de estas comprobaciones en `assert` ejecutables: no
son comentarios que digan "esto debería cumplirse", son afirmaciones que
detienen el notebook si no se cumplen.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Un data lake sin catálogo también tiene reglas</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code> inmutable es la referencia estable frente a la que se valida todo lo derivado</li><li style='margin:10px 0;list-style-position:outside'>Un contrato documentado dice rutas, columnas derivadas, métricas y regla de regeneración; <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>_SUCCESS</code> no es un dato</li><li style='margin:10px 0;list-style-position:outside'>Escribir Parquet correctamente no basta: la fiabilidad son las comprobaciones, y aquí serán <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>assert</code>, no comentarios</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Por qué particionar un data lake de ficheros</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Un data lake no tiene motor de base de datos con índices propios: sólo hay ficheros en HDFS</li><li style='margin:10px 0;list-style-position:outside'>Sin más información, responder «ventas de enero de 1999» obliga a abrir todos los ficheros de la tabla</li><li style='margin:10px 0;list-style-position:outside'>Se puede ahorrar lectura en tres niveles: columnas (formato columnar), grupos de filas (<em>predicate pushdown</em>) y ficheros enteros (particionado)</li><li style='margin:10px 0;list-style-position:outside'>Particionar es el ahorro más barato: se decide con el nombre del directorio, sin abrir ningún fichero</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Por qué particionar un data lake de ficheros

En una base de datos, el propio servidor gestiona dónde guardan las filas,
mantiene índices y estadísticas, y sabe ir directo a lo que pide una
consulta. En el data lake que construimos no hay nada de eso: hay
directorios y ficheros Parquet en HDFS, y los motores que los consultan
(Spark ahora, Trino en S6) sólo ven una lista de ficheros. Si esa lista no
trae ninguna organización, para responder «ventas de enero de 1999» el
motor no tiene más remedio que abrir todos los ficheros de la tabla y
comprobar qué filas cumplen la condición. Con SF3 es asumible; con una tabla
de cientos de gigabytes, no.

Hay tres niveles en los que un motor puede evitar leer lo que no necesita,
del más fino al más grueso:

- **Columnas.** Parquet es columnar: si la consulta sólo necesita dos
  columnas, el lector no toca las demás (*projection pushdown*, que ya
  viste en S4).
- **Grupos de filas dentro de un fichero.** Parquet guarda el mínimo y el
  máximo de cada columna por grupo de filas; el motor entrega la condición
  al lector (*predicate pushdown*, `PushedFilters` en el plan) y éste se
  salta los grupos que no pueden cumplirla. Pero hay que abrir cada
  fichero, al menos para leer su pie, y sólo funciona si los valores
  están agrupados: si cada grupo contiene valores de todo el rango, el
  mínimo y el máximo no descartan nada.
- **Ficheros enteros.** Es lo que hace el particionado: agrupar las filas
  en directorios según el valor de una o varias columnas, de modo que
  una consulta que filtra por esa columna sólo lista y abre los
  directorios que corresponden (*partition pruning*, `PartitionFilters`
  en el plan). Es el ahorro más barato, porque se decide con el nombre de
  la ruta, sin abrir un solo fichero, y se combina con los otros dos
  dentro de los ficheros que sí hay que leer.

El resto de la sección explica cómo funciona y cuándo conviene, y también
hasta dónde llega: el particionado por directorios es una herramienta
útil pero limitada, y entender sus límites es lo que explica por qué
existen los formatos de tabla como Iceberg.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Qué significa particionar</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Convención de Hive: directorios <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>columna=valor</code>, anidables; aquí, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_year=.../sold_month=...</code></li><li style='margin:10px 0;list-style-position:outside'>Una consulta que se limita a unos valores de esas columnas no abre los demás directorios (<em>partition pruning</em>)</li><li style='margin:10px 0;list-style-position:outside'>Los valores viven en la ruta, no dentro de los Parquet; cada fichero conserva su propio esquema y estadísticas</li><li style='margin:10px 0;list-style-position:outside'>Spark descubre <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_year</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_month</code> a partir de la ruta, sin metastore</li><li style='margin:10px 0;list-style-position:outside'>No es una partición de ejecución de Spark, ni un bloque HDFS, ni un <em>split</em>, ni una partición Iceberg</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Qué significa particionar

Particionar datos consiste en agrupar físicamente filas relacionadas y
escribirlas en directorios que codifican valores de columnas. Para la tabla
que vamos a construir, la estructura de directorios tendrá esta forma:

```text
web_sales/
├── sold_year=1998/
│   ├── sold_month=1/
│   │   └── part-....parquet
│   ├── sold_month=2/
│   │   └── part-....parquet
│   └── ...
├── sold_year=1999/
│   └── ...
└── ...
```

Esta organización viene de Apache Hive, que en HDFS guardaba cada
partición de una tabla en un subdirectorio `columna=valor`. Se ha
convertido en una convención que leen sin más Spark, Trino, DuckDB, Polars
o PyArrow, y por eso se habla de particionado *estilo Hive* aunque no haga
falta Hive para usarlo. Tiene tres rasgos que conviene tener claros:

- **Niveles anidados.** Cada columna de partición añade un nivel de
  directorios, en el orden en que se declara: aquí `sold_year` primero y
  `sold_month` dentro. Una consulta por año lee un único subárbol;
  una consulta por año y mes lee un único directorio hoja.
- **El valor está en la ruta, no en los ficheros.** Spark no escribe
  `sold_year` ni `sold_month` dentro de los Parquet de cada directorio: el
  valor se deduce del nombre del directorio, y por eso aparece como columna
  al leer la ruta raíz.
- **Cada fichero es autónomo.** Cada Parquet lleva en su pie su propio
  esquema y las estadísticas de sus grupos de filas, de modo que no hay un
  fichero central que describa la tabla. Eso permite escribir o reemplazar
  una partición sin tocar las demás, pero también significa que nada
  obliga a que todas tengan el mismo esquema: si una partición antigua
  y otra nueva difieren, Spark necesita la opción `mergeSchema` para
  reconciliarlas.

Una consulta que se limita a un `sold_year` (y, más aún, a un `sold_year` y
un `sold_month`) puede evitar leer directorios completos que no contienen
datos relevantes: es la poda de particiones (*partition pruning*). Spark puede descubrir esas columnas a partir de la ruta,
incluso sin un Hive Metastore de por medio: al leer la ruta raíz de la
tabla, `sold_year` y `sold_month` aparecen como columnas normales del
DataFrame aunque no estén dentro de ningún fichero Parquet. Cuando haga
falta indicar explícitamente desde qué directorio raíz debe realizarse ese
descubrimiento —por ejemplo, al leer un único subdirectorio de partición—
existe la opción `basePath`.

Este particionado físico por directorios **no debe confundirse** con otras
cosas que también se llaman "partición": algunas ya han aparecido en el
curso y otras llegarán más adelante:

- las **particiones de ejecución de Spark**, la unidad sobre la que trabaja
  cada tarea del motor (S4, sección "Particiones de ejecución");
- las **particiones de ventana**, el grupo de filas sobre el que opera una
  función de ventana (`Window.partitionBy`, S4);
- los **bloques HDFS**, la unidad de almacenamiento físico de bytes en el
  sistema de ficheros (S1);
- los ***splits*** que un motor SQL crea al planificar una consulta sobre
  un conjunto de ficheros, que veremos con Trino en S6;
- las **particiones lógicas y transformaciones ocultas** que tendrá una
  tabla Iceberg (`month()`, `year()`...), que veremos en S7 y que usarán la
  misma columna de fecha derivada que estamos a punto de construir, pero
  gestionadas por el propio formato de tabla en lugar de por el nombre de
  un directorio.

Ampliando las filas de particiones de la tabla "Capas que se deben
distinguir" de S4:

| Concepto | Significado |
| --- | --- |
| Partición de ejecución de Spark | Porción lógica de un DataFrame que procesa una tarea; depende del paralelismo, no de cómo están organizados los ficheros en disco |
| Partición de ventana (S4) | Grupo de filas sobre el que opera una función de ventana (`Window.partitionBy`); no cambia cómo se leen ni cómo se guardan los datos |
| Bloque HDFS | Unidad de almacenamiento físico de bytes; no tiene por qué coincidir con una fila, una columna ni un fichero completo |
| *Split* de un motor SQL (S6) | Unidad de trabajo en que un motor como Trino divide la lectura de los ficheros al planificar una consulta; se parece a la partición de ejecución de Spark, pero es de otro motor |
| Partición Hive (esta sesión) | Directorio `columna=valor` que agrupa físicamente filas por el valor de una o varias columnas |
| Partición/transformación Iceberg (S7) | Agrupación lógica gestionada por el formato de tabla, que puede usar transformaciones como `month()` sin exigir una columna física propia |

Compartir la palabra "partición" no las convierte en la misma unidad; a lo
largo de esta sesión seguiremos usando siempre "partición Hive" o
"directorio `columna=valor`" cuando nos refiramos específicamente a lo que
vamos a construir aquí.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Límites del particionado por directorios</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><strong>Una sola organización:</strong> los datos se agrupan por un único criterio a la vez; las consultas por otro atributo, como el cliente, siguen abriendo todo</li><li style='margin:10px 0;list-style-position:outside'><strong>Más columnas, más directorios:</strong> cada nivel multiplica el número de directorios; sólo sirven columnas de pocos valores distintos</li><li style='margin:10px 0;list-style-position:outside'><strong>Hay que filtrar por la columna de partición:</strong> un filtro por <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_sold_date</code> no poda, porque Spark no sabe que la fecha determina el año y el mes</li><li style='margin:10px 0;list-style-position:outside'><strong>Cambiarlo es reescribir:</strong> pasar de mes a día obliga a reescribir los datos y a avisar a quien lea por ruta</li><li style='margin:10px 0;list-style-position:outside'><strong>Qué hacer entonces:</strong> ordenar dentro de los ficheros, <em>bucketing</em>, copias con otra partición o un formato de tabla como Iceberg (S7)</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

### Límites del particionado por directorios

El particionado estilo Hive es sencillo y funciona, pero tiene límites que
conviene conocer antes de apoyarse en él:

- **Sólo hay una organización física.** Las filas están en un único árbol
  de directorios, así que los datos quedan agrupados por un criterio (aquí,
  el tiempo) y por ningún otro. Las consultas que filtran por otro
  atributo no se benefician: ya lo anticipó la consulta 5, el historial de
  compras de un cliente (`SELECT ... FROM web_sales_silver WHERE
  ws_bill_customer_sk = <cliente>`), que seguirá abriendo todos los ficheros,
  y lo comprobaremos en los planes.
- **Más de un atributo no sale gratis.** Si se quiere particionar por
  varios, hay que anidar más niveles, y el número de directorios es el
  producto de los valores distintos de cada columna: año × mes × región
  puede dar cientos de directorios con muy pocos datos, el problema de los
  ficheros pequeños. Por eso sólo valen columnas con pocos valores
  distintos, y las de muchos valores (cliente, artículo, pedido) quedan
  descartadas. Además, el orden de los niveles importa: los datos quedan
  agrupados sobre todo por el primero (aquí, el año), y el segundo sólo
  subdivide cada grupo.
- **La partición es un dato visible.** Para que se pode, la consulta tiene
  que filtrar por la columna de partición (`sold_year`, `sold_month`), no
  por la fecha de la que se derivó (`ws_sold_date`). Quien consulta tiene
  que conocer esas columnas auxiliares, y el motor no puede deducir la
  relación por sí solo.
- **Cambiar el esquema de particiones es reescribir.** Pasar de mes a día,
  o añadir una columna, significa reescribir todos los datos y avisar a
  todo lo que lea por la ruta antigua.
- **Los directorios no son una tabla.** El motor lista los directorios en
  cada consulta, y no hay transacciones ni versiones: quien lee mientras se
  escribe puede ver una partición a medio hacer (lo recuerda la sección
  «Buenas prácticas que quedan pendientes del catálogo», al final de esta
  sesión).

¿Qué se hace entonces cuando una consulta necesita otro atributo?
Hay varias salidas, ninguna perfecta:

- **Ordenar los datos** por ese atributo dentro de los ficheros, para que
  las estadísticas de Parquet (mínimo y máximo por grupo de filas)
  permitan saltarse la mayoría con *predicate pushdown*. No crea
  directorios, pero sólo sirve si la ordenación se mantiene.
- **Agrupar en cubos** (*bucketing*): repartir las filas en un número fijo
  de ficheros según un hash de la columna, útil para columnas de muchos
  valores como el cliente.
- **Mantener copias** de la tabla con otra organización: en `silver`, una
  segunda tabla con las mismas filas y su propio contrato, a costa de
  duplicar espacio y de tener que mantenerlas sincronizadas.
- **Usar un formato de tabla** que gestione el particionado por ti.
  Iceberg lo hace: declara las particiones como transformaciones de las
  columnas (`month(ws_sold_date)`, `bucket(N, ws_bill_customer_sk)`...),
  de modo que quien consulta filtra por la fecha sin saber cómo está
  particionada la tabla, permite combinar varios criterios, y permite
  cambiar el esquema de particiones sin reescribir lo ya escrito. S7
  retoma exactamente estos límites —transformaciones de partición ocultas,
  ordenación y ficheros pequeños— sobre la misma tabla `web_sales`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Elegir bien la columna de partición</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><strong>Primero, las consultas:</strong> de las que esperamos sobre la tabla salen las columnas de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> y las candidatas a partición</li><li style='margin:10px 0;list-style-position:outside'><strong>Criterios:</strong> consultas que se limitan a menudo a esa columna, pocos valores distintos, directorios grandes, ficheros útiles</li><li style='margin:10px 0;list-style-position:outside'><strong>Cardinalidades reales de SF3:</strong> pedidos, artículos y clientes tienen muchísimos valores; años y meses, unas decenas</li><li style='margin:10px 0;list-style-position:outside'><strong>Lo que usamos para dividir:</strong> <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_year</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_month</code>, en ese orden, derivadas de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_sold_timestamp</code></li><li style='margin:10px 0;list-style-position:outside'><strong>Lo que descartamos:</strong> las claves, la marca de tiempo y la fecha completa, por demasiados directorios</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Diseño de las particiones

Las columnas de partición deben elegirse a partir de las consultas y del
tamaño de los datos, no por costumbre. Primero los criterios, después lo que
usamos en `web_sales` y por qué descartamos el resto.

### Criterios para elegir una columna de partición

Una buena columna de partición cumple, a la vez:

- **Las consultas se limitan a ella.** La mayoría de las consultas
  esperadas filtran por esa columna; si no, la poda casi nunca se activa.
- **Pocos valores distintos.** Ni una única partición gigante, ni miles de
  particiones diminutas.
- **Directorios grandes.** Cada directorio debe tener datos suficientes
  para que merezca la pena evitar leerlo.
- **Fácil de explicar.** Debe poder describirse en una frase a quien vaya a
  consultar la tabla.
- **Pocos ficheros pequeños.** Es el mismo problema que ya vimos en S4 con
  `repartition` frente a `coalesce`.

### Qué usamos para dividir `web_sales`

- **Columnas de partición:** `sold_year` y `sold_month`, en ese orden. No
  existen en `raw`: las derivamos de `ws_sold_timestamp`, y `silver` las
  añade como columnas nuevas.
- **Por qué el tiempo:** las consultas 1 a 4 se limitan a un año, a unos
  meses o a un trimestre ("ventas de 2001", "ventas de enero de 1999"), y
  se explica sin saber nada más del dominio.
- **Qué directorios resultan:** una carpeta por año, y dentro una por mes.
  Las 2.160.165 líneas de SF3 se reparten entre unas sesenta combinaciones
  de año y mes: de media, unas 35.000 líneas por mes.

### Qué descartamos y por qué

- **`ws_bill_customer_sk`, `ws_item_sk` y `ws_order_number`:** identifican
  entidades (clientes, artículos y pedidos) con muchísimos valores
  distintos, muy por encima del número de años o meses que aparecen en las
  ventas. Particionar por una de ellas crearía un directorio por cliente,
  artículo o pedido, casi todos con uno o muy pocos ficheros diminutos:
  exactamente el problema de los "ficheros pequeños" que penaliza tanto a
  HDFS (metadatos del NameNode por fichero) como a cualquier motor que
  después tenga que abrir todos esos directorios. Comprobaremos estas
  cardinalidades con datos reales de SF3 en la sección siguiente.
- **`ws_sold_timestamp`:** tiene casi un valor distinto por pedido, el
  mismo problema que las claves.
- **La fecha sin la hora (`ws_sold_date`):** produciría un directorio por
  día, más de mil entre el 2 de enero de 1998 y el 2 de enero de 2003, el
  periodo de ventas de SF3. Por eso derivamos `sold_year` y `sold_month` en
  lugar de particionar directamente por la fecha.
- **Sólo `sold_year`:** es la alternativa más gruesa. Elegir entre ella y
  la pareja `sold_year, sold_month` debe relacionarse con el tamaño real de
  los fragmentos resultantes, no con una preferencia a priori: más niveles
  de partición no significan automáticamente más rendimiento, sobre todo
  si cada partición mensual termina teniendo un único fichero pequeño.
  Para SF3 usaremos la pareja y, más abajo, mediremos qué tamaño tienen
  los ficheros que resultan.

### Qué tamaño tendrán las particiones

La media de unas 35.000 líneas por mes esconde mucha estacionalidad: unas
20.000 de enero a julio, unas 46.000 de agosto a octubre y más de 70.000 en
noviembre y diciembre. Qué granularidad conviene depende además de la
escala del dataset: con SF1000 cada mes pesaría unas trescientas veces más.

De 2003 sólo hay ventas del 1 y el 2 de enero, porque ahí terminan los
datos de TPC-DS. Así que `sold_year=2003` tendrá un único directorio,
`sold_month=1`, con unas 4.800 líneas y un fichero mucho más pequeño que
los demás. Lo veremos al listar los directorios.

## Ejercicio de transformación con Spark

El ejercicio crea una salida nueva a partir de los Parquet de S2, sin
tocarlos:

```text
/datalake/raw/tpcds/web_sales
    → /datalake/silver/tpcds/web_sales/sold_year=.../sold_month=.../part-...
```

El flujo, celda a celda, será: leer `web_sales` desde su ruta física;
enumerar las consultas que esperamos sobre `silver`, que deciden sus
columnas y sus candidatas a partición; comprobar con datos reales la
cardinalidad de esas candidatas; conservar las columnas de negocio necesarias y derivar de
`ws_sold_timestamp` la fecha, `sold_year` y `sold_month`; validar filas y
fechas con `assert`; escribir la salida
particionada; escribir además una salida de control sin particionar; y
agregar desde `silver` los pedidos, las líneas y el beneficio por año para
escribir el producto `gold`. Después se relee `silver`, se comparan los
planes, los ficheros leídos y los tiempos de una lectura completa y de las
consultas de un año y de un cliente, y un ejercicio construye una segunda
tabla de laboratorio organizada por cliente.

### Leer la tabla fuente

In [ ]:
# Qué significa cada columna de web_sales, la tabla fuente de silver.
%esquema web_sales

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.functions import col, count, countDistinct, month, to_date, when, year

web_sales_path: str = "/datalake/raw/tpcds/web_sales"
web_sales: DataFrame = spark.read.parquet(web_sales_path)

print(f"Ruta física leída: {web_sales_path}")
web_sales.printSchema()

### Qué consultas esperamos sobre `silver`

Ni las columnas de `silver` ni sus particiones se eligen en abstracto:
salen de las preguntas que se le van a hacer a la tabla. Antes de decidir
nada, las escribimos. Para `web_sales` tomamos tres preguntas nuevas sobre
el tiempo y las de negocio de S2:

| # | Consulta | A qué parte de la tabla se limita | Columnas de `web_sales` que necesita |
| --- | --- | --- | --- |
| 1 | Evolución mensual de un año: pedidos, ventas, importe pagado y beneficio de cada mes | Un año | `ws_sold_timestamp`, `ws_order_number`, `ws_ext_sales_price`, `ws_net_paid`, `ws_net_profit` |
| 2 | Campaña de Navidad: noviembre y diciembre de un año frente al resto | Un año y dos de sus meses | Las de la consulta 1 |
| 3 | Artículos más vendidos de un trimestre | Tres meses de un año | `ws_sold_timestamp`, `ws_item_sk`, `ws_quantity`, `ws_ext_sales_price` |
| 4 | Descuento medio de cada mes frente al precio de lista | Un año | `ws_sold_timestamp`, `ws_quantity`, `ws_list_price`, `ws_ext_discount_amt` |
| 5 | Historial de compras de un cliente (en S2, el cliente con más pedidos) | Un cliente | `ws_bill_customer_sk`, `ws_order_number`, `ws_sold_timestamp`, `ws_net_paid` |
| 6 | Compras por región de facturación y distribución horaria de las compras (S2) | Ninguna: toda la tabla | `ws_bill_addr_sk` (para unir con `customer_address`), `ws_sold_timestamp` (su hora) |

Las mismas seis consultas en SQL, sobre `web_sales_silver` (el nombre que
daremos en SQL a la tabla refinada), con un año de ejemplo y `<cliente>` como marcador; en
el notebook los elegiremos con datos reales. Las consultas 1 a 4 se limitan
a un periodo; la 5, a un cliente; la 6 no filtra:

```sql
-- 1. Evolución mensual de un año
SELECT sold_month,
       COUNT(DISTINCT ws_order_number) AS pedidos,
       SUM(ws_ext_sales_price)         AS ventas,
       SUM(ws_net_paid)                AS pagado,
       SUM(ws_net_profit)              AS beneficio
FROM web_sales_silver
WHERE sold_year = 2001
GROUP BY sold_month ORDER BY sold_month;

-- 2. Campaña de Navidad: noviembre y diciembre de un año
SELECT sold_month,
       COUNT(DISTINCT ws_order_number) AS pedidos,
       SUM(ws_net_paid)                AS pagado,
       SUM(ws_net_profit)              AS beneficio
FROM web_sales_silver
WHERE sold_year = 2001 AND sold_month IN (11, 12)
GROUP BY sold_month ORDER BY sold_month;

-- 3. Artículos más vendidos de un trimestre
SELECT ws_item_sk, SUM(ws_quantity) AS unidades, SUM(ws_ext_sales_price) AS ventas
FROM web_sales_silver
WHERE sold_year = 2001 AND sold_month BETWEEN 1 AND 3
GROUP BY ws_item_sk ORDER BY unidades DESC LIMIT 10;

-- 4. Descuento medio de cada mes frente al precio de lista
SELECT sold_month,
       SUM(ws_ext_discount_amt) / SUM(ws_list_price * ws_quantity) AS descuento_medio
FROM web_sales_silver
WHERE sold_year = 2001
GROUP BY sold_month ORDER BY sold_month;

-- 5. Historial de compras de un cliente (el que más pedidos tiene en S2)
SELECT ws_order_number, ws_sold_timestamp, ws_net_paid
FROM web_sales_silver
WHERE ws_bill_customer_sk = <cliente>
ORDER BY ws_sold_timestamp;

-- 6. Compras por región (provincia) de facturación y distribución horaria (toda la tabla)
SELECT ca.ca_county AS provincia, COUNT(*) AS lineas
FROM web_sales_silver ws
JOIN customer_address ca ON ws.ws_bill_addr_sk = ca.ca_address_sk
GROUP BY ca.ca_county;

SELECT hour(ws_sold_timestamp) AS hora, COUNT(*) AS lineas
FROM web_sales_silver
GROUP BY hour(ws_sold_timestamp) ORDER BY hora;
```

De esta lista salen dos decisiones del contrato:

- **Qué columnas conserva `silver`.** Juntas, las seis consultas usan
  exactamente las 11 columnas de `web_sales` que declara el contrato, y
  `silver` añade `ws_sold_date`, `sold_year` y `sold_month`, derivadas de la
  marca de tiempo. Ninguna consulta usa las otras 22, que se quedan en
  `raw`. Si mañana se espera una consulta sobre costes de envío, o se lee
  `raw`, o se cambia el contrato, y con él los esquemas y las sesiones que
  dependen de ellos.
- **Qué columnas son candidatas a partición.** Son aquellas a las que las
  consultas se limitan a menudo: el tiempo (consultas 1 a 4) y el cliente
  (consulta 5). La consulta 6 recorre la tabla entera, y ninguna partición
  la ayuda. Entre las candidatas decide la cardinalidad, que comprobamos a
  continuación: una partición sólo es útil si tiene pocos valores distintos
  y cada uno reúne muchas filas.

Hay además una tercera decisión que en esta sesión dejamos sin tomar:
**cuántas tablas tiene `silver`**. La zona refinada no tiene por qué
contener una sola tabla por cada tabla de `raw`. Si una consulta frecuente
necesita los mismos datos organizados por otra clave —aquí, la consulta 5,
el historial de compras de un cliente (`WHERE ws_bill_customer_sk = ...`),
frente a las consultas 1 a 4, por tiempo—, una opción es
generar una segunda tabla en `silver` con las mismas filas (o sólo con las
columnas que esa consulta usa), organizada de otra manera: por ejemplo,
ordenada o agrupada en cubos por `ws_bill_customer_sk`, bajo una ruta como
`/datalake/silver/tpcds/web_sales_by_customer`. Cada tabla llevaría su
propio contrato (origen, esquema, particionado, regeneración) y se
reconstruiría desde `raw` con la misma regla. El precio es duplicar espacio
y tener más salidas que mantener sincronizadas. En la transformación
principal de esta sesión no lo hacemos: construimos una única tabla de
`silver`, particionada por tiempo, y medimos lo que le cuesta a la consulta
5. Más abajo, un ejercicio te pide construir esa segunda tabla, pero como
tabla de laboratorio en `/user/luser/s5`, fuera del contrato, para medir
cuánto se gana. La salida de control sin particionar que escribiremos más
abajo tampoco es esto: sólo sirve para comparar.

### Comprobar las cardinalidades antes de elegir la partición

Antes de transformar nada, medimos con datos reales de SF3 cuántos valores
distintos tienen el pedido, el artículo y el cliente, tres claves que usan
las consultas anteriores y que «Diseño de las particiones» descartaba como
partición principal.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Antes de ejecutar la celda: ¿cuántos directorios crearía particionar por <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_bill_customer_sk</code>? ¿Y por <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_order_number</code>?</li><li style='margin:10px 0;list-style-position:outside'>¿Cuántas filas tendría, de media, cada directorio?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

**Equivalente en SQL.** A partir de aquí, antes de cada consulta de Spark escribimos su equivalente en SQL, para poder leerla en cualquiera de los dos lenguajes. No se ejecuta aquí: en S4 se podía ejecutar registrando el DataFrame con `createOrReplaceTempView`, y en S6 las tablas tendrán nombre en el metastore. Usamos `web_sales` para la lectura de `raw` y `web_sales_silver` para la tabla refinada. Esta consulta cuenta los valores distintos de las tres claves:

```sql
SELECT COUNT(DISTINCT ws_order_number)     AS pedidos,
       COUNT(DISTINCT ws_item_sk)          AS articulos_distintos,
       COUNT(DISTINCT ws_bill_customer_sk) AS clientes_distintos
FROM web_sales;
```

In [ ]:
cardinality_report: DataFrame = web_sales.select(
    countDistinct("ws_order_number").alias("pedidos"),
    countDistinct("ws_item_sk").alias("articulos_distintos"),
    countDistinct("ws_bill_customer_sk").alias("clientes_distintos"),
)
cardinality_report.show()

Compara esas tres cifras con el número de años y meses distintos que
veremos más abajo tras derivar las particiones (a lo sumo unas pocas decenas de
combinaciones `sold_year`/`sold_month`). Cualquiera de las tres columnas
anteriores generaría muchísimos más directorios que la pareja
`sold_year`/`sold_month`, la mayoría con muy pocas filas: es precisamente
la razón por la que no las usamos como partición principal.

La consulta 5 es la que sale perdiendo: se limita a un cliente tan a
menudo como las consultas 1 a 4 a un periodo, pero particionar por
`ws_bill_customer_sk` crearía un directorio por cliente, más de cien mil
en SF3, casi todos con un puñado de líneas. Esa consulta necesita otra
organización: ordenar o agrupar las filas por cliente, para que las
estadísticas de Parquet permitan saltarse la mayor parte de cada fichero,
algo que veremos con Iceberg en S7. En un data lake de ficheros, la otra
salida es la segunda tabla de `silver` que se describió arriba, con las
mismas filas organizadas por cliente; la construirás en un ejercicio más
abajo. Al comparar los planes, más abajo, comprobaremos que con la partición por año y mes la consulta 5 sigue
abriendo todos los ficheros.

### Derivar la fecha y construir el esquema *silver*

Nos quedamos con las líneas que tienen fecha, conservamos las columnas de
negocio del contrato y derivamos de `ws_sold_timestamp` tres columnas:
`ws_sold_date`, la fecha sin la hora, con `to_date`; y las particiones
`sold_year` y `sold_month`, con `year` y `month`.

**Equivalente en SQL.** La transformación que construye `silver` es una consulta de selección con columnas derivadas y un filtro:

```sql
SELECT ws_order_number, ws_item_sk, ws_bill_customer_sk, ws_bill_addr_sk,
       ws_sold_timestamp,
       to_date(ws_sold_timestamp) AS ws_sold_date,
       ws_quantity, ws_list_price, ws_ext_discount_amt, ws_ext_sales_price,
       ws_net_paid, ws_net_profit,
       year(ws_sold_timestamp)    AS sold_year,
       month(ws_sold_timestamp)   AS sold_month
FROM web_sales
WHERE ws_sold_timestamp IS NOT NULL;
```

In [ ]:
web_sales_silver: DataFrame = web_sales.filter(col("ws_sold_timestamp").isNotNull()).select(
    "ws_order_number",
    "ws_item_sk",
    "ws_bill_customer_sk",
    "ws_bill_addr_sk",
    "ws_sold_timestamp",
    to_date("ws_sold_timestamp").alias("ws_sold_date"),
    "ws_quantity",
    "ws_list_price",
    "ws_ext_discount_amt",
    "ws_ext_sales_price",
    "ws_net_paid",
    "ws_net_profit",
    year("ws_sold_timestamp").alias("sold_year"),
    month("ws_sold_timestamp").alias("sold_month"),
)
web_sales_silver.printSchema()

### Comprobar el esquema declarado

`printSchema()` muestra el esquema que ha resultado de la
transformación; falta compararlo con el que declara el contrato. La
celda siguiente compara, columna a columna y en orden, el nombre y el
tipo; no la nulabilidad, por lo explicado en «Los esquemas de `silver`
y `gold`». Si una columna sobra, falta o ha cambiado de tipo, el
mensaje del `assert` la nombra:

In [ ]:
silver_columns: list[tuple[str, str]] = [
    (field.name, field.dataType.simpleString()) for field in web_sales_silver.schema.fields
]
silver_contract_columns: list[tuple[str, str]] = [
    (field.name, field.dataType.simpleString()) for field in web_sales_silver_schema.fields
]

assert silver_columns == silver_contract_columns, (
    "silver no cumple su esquema declarado (si las dos listas están vacías, "
    "cambia el orden de las columnas): "
    f"no declaradas {sorted(set(silver_columns) - set(silver_contract_columns))}, "
    f"declaradas y ausentes {sorted(set(silver_contract_columns) - set(silver_columns))}"
)
print(f"silver cumple su esquema declarado: {len(silver_columns)} columnas.")

Hoy la comprobación pasa sin más, porque el `select` de la celda anterior
se escribió a partir del contrato. Su utilidad llega después: si alguien
cambia ese `select`, o si la fuente `raw` cambia el tipo de una columna,
el notebook se detiene aquí en lugar de escribir en `silver` una tabla
distinta de la que esperan S6, S7 y S8.

### Validar filas y fechas antes de escribir

Una transformación mal planteada puede perder más filas de las esperadas.
TPC-DS introduce deliberadamente una pequeña fracción de líneas de venta
sin fecha —una característica documentada del propio generador, pensada
para simular datos de origen imperfectos, no un error de esta sesión—:
esas líneas no se pueden asignar a ningún año ni mes, y el filtro
`isNotNull` las deja fuera de `silver`. Antes de escribir nada comprobamos
que la única pérdida de filas es exactamente esa, ni una más ni una menos:

**Equivalente en SQL.** Los tres recuentos son tres consultas:

```sql
SELECT COUNT(*) FROM web_sales;                                   -- raw_count
SELECT COUNT(*) FROM web_sales_silver;                            -- silver_count
SELECT COUNT(*) FROM web_sales WHERE ws_sold_timestamp IS NULL;   -- missing_date_count
```

In [ ]:
raw_count: int = web_sales.count()
silver_count: int = web_sales_silver.count()
missing_date_count: int = web_sales.filter(col("ws_sold_timestamp").isNull()).count()

print(f"Filas en la fuente (web_sales): {raw_count}")
print(f"Filas sin ws_sold_timestamp (no se pueden fechar): {missing_date_count}")
print(f"Filas en silver: {silver_count}")

assert raw_count - silver_count == missing_date_count, (
    "silver sólo debería perder las líneas sin ws_sold_timestamp, ninguna otra: "
    f"raw={raw_count} silver={silver_count} sin_fecha={missing_date_count}"
)

Después comprobamos que ni la marca de tiempo ni las columnas derivadas de
ella aparecen nulas en ninguna fila de `silver`. Hoy esta comprobación no
puede fallar: el `filter` de la transformación ya ha descartado las
líneas sin marca de tiempo, y `to_date`, `year` y `month` de una marca no
nula nunca dan nulo. Como la del esquema, protege frente a un cambio
futuro: si alguien quita ese `filter` (es la primera actividad de más
abajo), esta celda también lo detecta.

**Equivalente en SQL.** Cuenta las filas de `silver` con alguna de las cuatro columnas de fecha nula:

```sql
SELECT COUNT(*)
FROM web_sales_silver
WHERE ws_sold_timestamp IS NULL OR ws_sold_date IS NULL
   OR sold_year IS NULL OR sold_month IS NULL;
```

In [ ]:
null_dates: int = web_sales_silver.filter(
    col("ws_sold_timestamp").isNull()
    | col("ws_sold_date").isNull()
    | col("sold_year").isNull()
    | col("sold_month").isNull()
).count()

print(f"Líneas con marca de tiempo, fecha, año o mes nulos: {null_dates}")

assert null_dates == 0, f"No debería haber fechas nulas en silver: {null_dates} filas afectadas"

Las claves se comprueban de dos formas distintas, según lo que dice el
contrato. `ws_order_number` y `ws_item_sk` identifican la línea: un nulo
en cualquiera de ellas es un error y detiene el notebook. Las claves de
unión con cliente y domicilio pueden venir vacías desde la fuente: no se
exige que sean cero, pero se cuentan, porque son las líneas que se
quedarán sin pareja en un `JOIN` con `customer` o `customer_address`. Una
sola agregación cuenta los nulos de las cuatro columnas:

**Equivalente en SQL.** Cuenta, para cada clave, las líneas en que es nula:

```sql
SELECT COUNT_IF(ws_order_number     IS NULL) AS ws_order_number,
       COUNT_IF(ws_item_sk          IS NULL) AS ws_item_sk,
       COUNT_IF(ws_bill_customer_sk IS NULL) AS ws_bill_customer_sk,
       COUNT_IF(ws_bill_addr_sk     IS NULL) AS ws_bill_addr_sk
FROM web_sales_silver;
```

In [ ]:
from pyspark.sql import Row

key_columns: list[str] = [
    "ws_order_number",
    "ws_item_sk",
    "ws_bill_customer_sk",
    "ws_bill_addr_sk",
]
null_keys: Row | None = web_sales_silver.select(
    *[count(when(col(name).isNull(), 1)).alias(name) for name in key_columns]
).first()

for name in key_columns:
    print(f"Líneas con {name} nulo: {null_keys[name]}")

assert null_keys["ws_order_number"] == 0 and null_keys["ws_item_sk"] == 0, (
    f"Las claves que identifican la línea no pueden ser nulas: {null_keys}"
)

Por último, comprobamos que los rangos de las columnas de partición tienen
sentido. Para `sold_month` el límite es universal: debe estar siempre entre
1 y 12, y un `assert` lo exige. Para `sold_year` no lo hay, porque depende
del periodo que cubren los datos: la celda imprime el año mínimo y el
máximo, y escribir su comprobación es una de las actividades de más abajo.

**Equivalente en SQL.** Los límites de año y mes, y el número de combinaciones distintas:

```sql
SELECT MIN(sold_year)  AS min_year,  MAX(sold_year)  AS max_year,
       MIN(sold_month) AS min_month, MAX(sold_month) AS max_month
FROM web_sales_silver;

SELECT COUNT(*) FROM (SELECT DISTINCT sold_year, sold_month FROM web_sales_silver);
```

In [ ]:
year_month_bounds: Row | None = web_sales_silver.select(
    F.min("sold_year").alias("min_year"),
    F.max("sold_year").alias("max_year"),
    F.min("sold_month").alias("min_month"),
    F.max("sold_month").alias("max_month"),
).first()

print(year_month_bounds)

assert year_month_bounds["min_month"] >= 1 and year_month_bounds["max_month"] <= 12, (
    f"sold_month debe estar en el rango 1-12: {year_month_bounds}"
)

# Las cifras que la sección anterior prometía comparar con las
# cardinalidades descartadas como partición principal.
year_month_combinations: int = web_sales_silver.select("sold_year", "sold_month").distinct().count()
print(f"Combinaciones distintas de sold_year/sold_month: {year_month_combinations}")

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">La transformación validada, no supuesta</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Las columnas, su orden y sus tipos son los del esquema declarado</li><li style='margin:10px 0;list-style-position:outside'>La única pérdida son las líneas sin <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_sold_timestamp</code>: ni una más, comprobado con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>assert</code></li><li style='margin:10px 0;list-style-position:outside'>Marca de tiempo, fecha, año y mes sin nulos en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code></li><li style='margin:10px 0;list-style-position:outside'>Las claves de la línea nunca nulas; las de cliente y domicilio, contadas porque pueden faltar</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_month</code> entre 1 y 12: los rangos de las columnas de partición también se comprueban</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Rompe una validación a propósito</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Quita el <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>filter</code> de las líneas sin fecha y ejecuta las comprobaciones: ¿cuál falla primero y qué dice su mensaje?</li><li style='margin:10px 0;list-style-position:outside'>Deriva <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_month</code> con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>dayofmonth</code> en lugar de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>month</code>: ¿qué <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>assert</code> lo detecta y por qué?</li><li style='margin:10px 0;list-style-position:outside'>Quita <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_net_paid</code> del <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>select</code>, o conviértela con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>.cast(&quot;double&quot;)</code>: ¿qué comprobación lo detecta y qué dice su mensaje?</li><li style='margin:10px 0;list-style-position:outside'>Escribe una comprobación nueva para <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_year</code>: ¿qué rango de años esperas y por qué?</li><li style='margin:10px 0;list-style-position:outside'>Antes de seguir, vuelve a ejecutar la celda original para restaurar <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales_silver</code></li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Escribir las tres salidas</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>repartition</code> por las columnas de partición antes de escribir: un fichero por directorio</li><li style='margin:10px 0;list-style-position:outside'>Una salida de control idéntica sin particionar, en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/user/luser/s5</code> para comparar</li><li style='margin:10px 0;list-style-position:outside'>El producto <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code>: pedidos, líneas y beneficio por año en un único fichero</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

### Escribir la salida *silver* particionada

Antes de escribir, reagrupamos las filas por las columnas de partición con
`repartition("sold_year", "sold_month")`: así todas las filas de cada
combinación de año y mes van a parar a la misma tarea, y cada directorio
`sold_year=.../sold_month=...` recibe exactamente un fichero, uno por
combinación (61 en SF3). Las tareas del *shuffle* son como mucho 8, el
`spark.sql.shuffle.partitions` que fijó la sesión, así que cada una
escribe varios directorios. Sin `repartition`, cada tarea escribiría un
fragmento en cada directorio cuyas filas le tocaran: en el peor caso,
tantos ficheros por directorio como tareas. Es la misma lección de "ficheros pequeños" de S4, aplicada
ahora a una escritura particionada. `mode("overwrite")` hace la escritura
idempotente: volver a ejecutar esta celda reconstruye la tabla desde cero
en lugar de acumular ficheros duplicados.

In [ ]:
silver_path: str = "/datalake/silver/tpcds/web_sales"

(
    web_sales_silver.repartition("sold_year", "sold_month")
    .write.mode("overwrite")
    .partitionBy("sold_year", "sold_month")
    .parquet(silver_path)
)

print(f"Escrito: {silver_path}")

### Escribir una salida de control sin particionar

Para poder comparar honestamente "qué aporta particionar" necesitamos una
versión de referencia con exactamente las mismas filas y columnas, pero
escrita en un único directorio sin subdirectorios `columna=valor`. Esta
salida es únicamente un artefacto de laboratorio para la comparación de
esta sesión: a diferencia de `/datalake/silver/tpcds/web_sales`, no forma
parte del contrato que usarán S6, S7 o S8, así que la dejamos bajo la
ruta de trabajo personal `/user/luser/s5`, no bajo `/datalake`.

Hay otra diferencia, que veremos al releer `silver`: en la salida de
control, `sold_year` y `sold_month` son columnas normales, guardadas
dentro de cada fichero Parquet; en `silver`, `partitionBy` las saca de
los ficheros y sólo quedan en el nombre de los directorios.

In [ ]:
!hdfs dfs -mkdir -p /user/luser/s5

In [ ]:
silver_control_path: str = "/user/luser/s5/web_sales_silver_sin_particionar"

web_sales_silver.write.mode("overwrite").parquet(silver_control_path)

print(f"Escrito (control, sin particionar): {silver_control_path}")

### Agregar el producto *gold*

El producto `gold` responde a una pregunta de negocio concreta:

> "pedidos, líneas de venta y beneficio neto por año",

y se calcula agregando desde
`silver`, no releyendo `raw`: la celda lee la tabla que acabamos de escribir
en `silver_path` y agrupa por `sold_year`, una columna que Spark reconstruye
a partir de los nombres de directorio (lo veremos con detalle al releer la
tabla, más abajo). Mantenemos la misma disciplina de métricas
que en S2: `count(DISTINCT ws_order_number)` cuenta pedidos, `count(*)`
cuenta líneas de venta, y ambas cifras se muestran por separado para que no
se confundan.

La lectura de `silver` pasa su esquema declarado con
`.schema(web_sales_silver_schema)`: quien consume `silver` la lee por su
contrato, sin que Spark deduzca los tipos del pie de un fichero ni de los
nombres de los directorios. Spark no abre los ficheros para comprobar ese
esquema al definir la lectura. Si un fichero guardara una columna con un
tipo distinto del declarado, la consulta fallaría al leerlo con un error
`PARQUET_COLUMN_DATA_TYPE_MISMATCH`; si le faltara una columna del
contrato, Spark la leería como `NULL` sin avisar. Por eso la comprobación
que importa es la que hicimos antes de escribir.

**Equivalente en SQL.** El producto `gold` es esta agregación sobre `silver`:

```sql
SELECT sold_year,
       COUNT(DISTINCT ws_order_number) AS pedidos,
       COUNT(*)                        AS lineas_de_venta,
       SUM(ws_net_profit)              AS beneficio_neto
FROM web_sales_silver
GROUP BY sold_year
ORDER BY sold_year;
```

In [ ]:
web_sales_silver_written: DataFrame = spark.read.schema(web_sales_silver_schema).parquet(
    silver_path
)

web_sales_by_year: DataFrame = (
    web_sales_silver_written.groupBy("sold_year")
    .agg(
        countDistinct("ws_order_number").alias("pedidos"),
        count("*").alias("lineas_de_venta"),
        F.sum("ws_net_profit").alias("beneficio_neto"),
    )
    .orderBy("sold_year")
)

web_sales_by_year.printSchema()
web_sales_by_year.show()

Dos rasgos de esta tabla no son errores de la agregación, sino de los
datos:

- **El beneficio neto sale negativo todos los años**, y lo es en tres de
  cada cinco líneas de venta. Es una propiedad de los datos sintéticos de
  TPC-DS, que no imitan los márgenes de una empresa real; lo que importa
  aquí es la métrica (la suma de `ws_net_profit`), no su signo.
- **2003 parece un desplome**: unos 400 pedidos frente a unos 36.000 en
  cada uno de los años anteriores. No lo es: los datos de TPC-DS terminan
  el 2 de enero de 2003. Quien consulta `gold` no ve el detalle, sólo una fila
  por año, y la compararía con las demás. Por eso el contrato dice qué
  periodo cubre esa fila: un producto `gold` también debe advertir de sus
  periodos incompletos.

En ese esquema, `pedidos` y `lineas_de_venta` aparecen con
`nullable = false`: un recuento nunca es nulo, y Spark lo sabe. El
contrato las declara nullables porque así las verá cualquiera que lea
el fichero, como comprobaremos al releer `gold`. Comparamos nombres,
orden y tipos, como en `silver`, antes de escribir:

In [ ]:
gold_columns: list[tuple[str, str]] = [
    (field.name, field.dataType.simpleString()) for field in web_sales_by_year.schema.fields
]
gold_contract_columns: list[tuple[str, str]] = [
    (field.name, field.dataType.simpleString()) for field in web_sales_by_year_schema.fields
]

assert gold_columns == gold_contract_columns, (
    "gold no cumple su esquema declarado (si las dos listas están vacías, "
    "cambia el orden de las columnas): "
    f"no declaradas {sorted(set(gold_columns) - set(gold_contract_columns))}, "
    f"declaradas y ausentes {sorted(set(gold_contract_columns) - set(gold_columns))}"
)
print(f"gold cumple su esquema declarado: {len(gold_columns)} columnas.")

El producto `gold` es pequeño (una fila por año) y no necesita particionado
físico: lo escribimos como un único fichero Parquet con `coalesce(1)`, la
misma técnica de consolidación que usamos en S4 para evitar múltiples
ficheros diminutos.

In [ ]:
gold_path: str = "/datalake/gold/tpcds/web_sales_by_year"

web_sales_by_year.coalesce(1).write.mode("overwrite").parquet(gold_path)

print(f"Escrito: {gold_path}")

Releemos `gold` sin pasarle ningún esquema, como lo leería alguien que
sólo conoce la ruta. El esquema que Spark deduce del fichero debe ser
exactamente el declarado, ahora también en la nulabilidad:

In [ ]:
web_sales_by_year_read_back: DataFrame = spark.read.parquet(gold_path)

web_sales_by_year_read_back.printSchema()
web_sales_by_year_read_back.show()

assert web_sales_by_year_read_back.schema == web_sales_by_year_schema, (
    f"El fichero de gold no tiene el esquema declarado: {web_sales_by_year_read_back.schema}"
)

`pedidos` y `lineas_de_venta` han pasado a `nullable = true`. Spark sí
anotó en el fichero que esas dos columnas no tienen nulos (en Parquet,
una columna `REQUIRED` en lugar de `OPTIONAL`), pero al leer un Parquet
trata todas las columnas como nullables. Es la razón por la que el
contrato no promete `nullable=False`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Tres rutas para el mismo dato</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/datalake/silver/tpcds/web_sales</code>: detalle enriquecido, particionado por <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_year</code>/<code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_month</code></li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/user/luser/s5/...</code>: control de laboratorio, fuera del contrato porque sólo sirve para esta comparación</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/datalake/gold/tpcds/web_sales_by_year</code>: una fila por año, sin particionado: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code> responde, no se organiza</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">La evidencia de que la poda funciona</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Releer por la ruta raíz: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_year</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_month</code> aparecen sin metastore de por medio</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>PartitionFilters</code> en el plan y el recuento de ficheros en HDFS: qué abre Spark de verdad, no lo que devuelve <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>inputFiles()</code></li><li style='margin:10px 0;list-style-position:outside'>El tiempo de un portátil no basta como evidencia: caches, carga y Docker lo dominan; se mide después, como complemento</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Releer la tabla *silver* desde la ruta raíz

Cerramos el ciclo de escritura releyendo `/datalake/silver/tpcds/web_sales`
exactamente igual que leeríamos cualquier otro Parquet: por su ruta raíz,
sin mencionar los subdirectorios de partición en ningún sitio.

In [ ]:
web_sales_silver_read_back: DataFrame = spark.read.parquet(silver_path)

web_sales_silver_read_back.printSchema()

`sold_year` y `sold_month` aparecen en el esquema como columnas normales
del DataFrame, aunque ningún fichero Parquet las contenga físicamente:
Spark las ha descubierto a partir de los nombres de directorio
`sold_year=.../sold_month=...` durante el listado de la ruta. Esto ocurre
sin ningún Hive Metastore de por medio; es exactamente el "descubrimiento
por convención de directorios" que describimos en la sección "Qué
significa particionar". En S6 este mismo descubrimiento pasará a estar
centralizado en un catálogo, en lugar de depender de que cada motor liste
la ruta por su cuenta.

Lo que Spark deduce de los ficheros y de los nombres de directorio debe
coincidir, columna a columna y también en nulabilidad, con el esquema
declarado en el contrato:

In [ ]:
assert web_sales_silver_read_back.schema == web_sales_silver_schema, (
    f"La tabla silver releída no tiene el esquema declarado: {web_sales_silver_read_back.schema}"
)
print("El esquema de silver releído desde HDFS es el declarado en el contrato.")

## Comparar la lectura completa con las consultas de un año y de un cliente

El tiempo total de una consulta SF3 en un portátil, dentro de un
contenedor, no es una evidencia fiable de que el particionado ayuda:
depende de cachés del sistema operativo, de la carga del equipo, de Docker
y de cuántas veces se haya ejecutado ya la misma consulta antes. Lo que sí
podemos observar de forma reproducible es el **plan físico** de la consulta
y el **número de ficheros que Spark decide abrir** antes de leer una sola
fila. Eso es lo que vamos a comparar primero; más abajo medimos también el
tiempo, pero sólo como indicio complementario.

Compararemos tres lecturas de `silver`: la completa; la de la consulta 1
de «Qué consultas esperamos sobre `silver`», que se limita a un año; y la
de la consulta 5, que se limita a un cliente. Primero elegimos, con datos
reales, el año: el que tenga más líneas de venta en la tabla `silver`.

**Equivalente en SQL.** Líneas por año, de más a menos (`show()` las muestra todas y `first()` se queda con la primera):

```sql
SELECT sold_year, COUNT(*) AS count
FROM web_sales_silver
GROUP BY sold_year
ORDER BY count DESC, sold_year;
```

In [ ]:
year_counts: DataFrame = (
    web_sales_silver_read_back.groupBy("sold_year")
    .count()
    .orderBy(col("count").desc(), col("sold_year"))
)
year_counts.show()

sample_year: int = year_counts.first()["sold_year"]
print(f"Año elegido: {sample_year}")

Y el cliente: el que tiene más pedidos, la misma pregunta que S2
respondió con DuckDB.

**Equivalente en SQL.** El cliente con más pedidos distintos:

```sql
SELECT ws_bill_customer_sk, COUNT(DISTINCT ws_order_number) AS pedidos
FROM web_sales_silver
WHERE ws_bill_customer_sk IS NOT NULL
GROUP BY ws_bill_customer_sk
ORDER BY pedidos DESC, ws_bill_customer_sk
LIMIT 1;
```

In [ ]:
top_customer: Row | None = (
    web_sales_silver_read_back.filter(col("ws_bill_customer_sk").isNotNull())
    .groupBy("ws_bill_customer_sk")
    .agg(countDistinct("ws_order_number").alias("pedidos"))
    .orderBy(col("pedidos").desc(), col("ws_bill_customer_sk"))
    .first()
)
sample_customer: int = top_customer["ws_bill_customer_sk"]
print(f"Cliente elegido: {sample_customer}, con {top_customer['pedidos']} pedidos")

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Con el año elegido: ¿qué fracción de los ficheros leerá la consulta de ese año?</li><li style='margin:10px 0;list-style-position:outside'>¿Y la consulta del cliente (la 5: historial de compras de un cliente)? ¿Lee menos ficheros o sólo devuelve menos filas?</li><li style='margin:10px 0;list-style-position:outside'>¿Y un filtro por <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_net_profit &gt; 0</code>?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

**Equivalente en SQL.** Las tres lecturas que vamos a comparar son estas (`<año>` y `<cliente>` son los valores elegidos en las celdas anteriores):

```sql
-- full_scan: lectura completa
SELECT * FROM web_sales_silver;

-- filtered: un año (consulta 1, sin la agregación)
SELECT * FROM web_sales_silver WHERE sold_year = <año>;

-- customer_query: un cliente (consulta 5, sin la ordenación ni las otras columnas)
SELECT * FROM web_sales_silver WHERE ws_bill_customer_sk = <cliente>;
```

In [ ]:
full_scan: DataFrame = web_sales_silver_read_back
filtered: DataFrame = web_sales_silver_read_back.filter(col("sold_year") == sample_year)
customer_query: DataFrame = web_sales_silver_read_back.filter(
    col("ws_bill_customer_sk") == sample_customer
)

### Plan físico de la lectura completa

In [ ]:
full_scan.explain("formatted")

### Plan físico de la consulta de un año

Busca en el bloque `Scan parquet` una sección `PartitionFilters`: si
aparece con la condición `sold_year = <año elegido>`, significa que Spark
ha aplicado el filtro **antes** de decidir qué directorios listar, no
después de leer todas las filas.

In [ ]:
filtered.explain("formatted")

### Plan físico de la consulta de un cliente

Compara el bloque `Scan parquet` de la consulta 5 con el anterior:
¿aparece `PartitionFilters`? Busca también `PushedFilters`, la lista de
condiciones que Spark entrega al lector de Parquet.

In [ ]:
customer_query.explain("formatted")

`PartitionFilters` aparece vacío: la condición no habla de ninguna
columna de partición, así que Spark tiene que listar y abrir todos los
directorios. La condición está en `PushedFilters`: el lector de Parquet
la recibe y puede saltarse los grupos de filas cuyas estadísticas (el
mínimo y el máximo de la columna) demuestren que no contienen a ese
cliente. Pero `silver` no está ordenada por cliente: cada grupo de filas
reúne clientes de casi todo el rango, y esas estadísticas apenas
descartan nada. Es lo que anticipaba la sección de cardinalidades: esta
consulta no se resuelve con particiones, sino ordenando los datos: lo
probarás en un ejercicio más abajo, tras medir los tiempos, con una segunda
tabla de laboratorio, y en S7 veremos cómo se hace dentro de una misma tabla con Iceberg.

### Ficheros efectivamente leídos

`explain()` dice qué *decide* Spark; para ver qué *ha leído* hay que
ejecutar las tres consultas y mirar sus métricas. La celda siguiente lanza
una acción sobre cada una.

**Equivalente en SQL.** Contar las filas de cada una de las tres lecturas:

```sql
SELECT COUNT(*) FROM web_sales_silver;
SELECT COUNT(*) FROM web_sales_silver WHERE sold_year = <año>;
SELECT COUNT(*) FROM web_sales_silver WHERE ws_bill_customer_sk = <cliente>;
```

In [ ]:
full_scan_rows: int = full_scan.count()
filtered_rows: int = filtered.count()
customer_rows: int = customer_query.count()

print(f"Filas de la lectura completa: {full_scan_rows}")
print(f"Filas de la consulta de un año (sold_year = {sample_year}): {filtered_rows}")
print(f"Filas de la consulta de un cliente ({sample_customer}): {customer_rows}")

assert filtered_rows < full_scan_rows

Abre la pestaña **SQL / DataFrame** de <http://localhost:4040> —la
interfaz web de Spark presentada en S4— y entra en las tres últimas
consultas. En el nodo `Scan parquet` de cada una aparecen, entre otras, las
métricas *number of files read*, *size of files read* y *number of
partitions read*: en la consulta de un año deben ser una fracción de las de
la lectura completa, y en la de un cliente, las mismas que en la completa.
Esa es la medida directa de la poda.

Como comprobación independiente, contamos también los ficheros que existen
en HDFS bajo cada ruta: los de la partición del año elegido deberían
coincidir con el *number of files read* de la consulta de ese año.
`DataFrame.inputFiles()` no sirve para esto: en una relación basada en ruta
(sin catálogo) devuelve siempre la lista completa de ficheros de la tabla,
aunque el plan aplique `PartitionFilters`.

Los cuenta `hdfs dfs -count <ruta>`, que recorre el árbol completo bajo la
ruta y escribe, en este orden, el número de directorios, el de ficheros,
el tamaño lógico total en bytes y la propia ruta. La función
`hdfs_count` de la celda siguiente devuelve esas tres cifras; guardamos
también el tamaño, que usaremos al final para calcular el tamaño medio de
los ficheros. El marcador `_SUCCESS` sólo está en la raíz de la tabla y
cuenta como fichero: se resta, porque no es un dato.

In [ ]:
def hdfs_count(path: str) -> tuple[int, int, int]:
    """Directorios, ficheros y bytes bajo path, según hdfs dfs -count."""
    fields: list[str] = subprocess.run(
        ["hdfs", "dfs", "-count", path], check=True, capture_output=True, text=True
    ).stdout.split()
    return int(fields[0]), int(fields[1]), int(fields[2])


_, silver_files, silver_total_bytes = hdfs_count(silver_path)
full_scan_files: int = silver_files - 1  # descarta _SUCCESS
_, filtered_files, _ = hdfs_count(f"{silver_path}/sold_year={sample_year}")

print(f"Ficheros de datos bajo la ruta completa: {full_scan_files}")
print(f"Ficheros bajo sold_year={sample_year}: {filtered_files}")

assert filtered_files < full_scan_files, (
    "La partición de un solo año debería tener estrictamente menos ficheros que "
    f"la tabla completa: completa={full_scan_files} filtrada={filtered_files}"
)
print(f"Ficheros descartados por la poda de particiones: {full_scan_files - filtered_files}")

`sample_year` es el año con más líneas de SF3, cuyas ventas van de 1998 a
2002 más los dos primeros días de 2003: la partición de cualquiera de
esos cinco años reúne 12 de los 61 ficheros, uno por mes. Por tanto
`filtered_files` debe ser estrictamente menor que `full_scan_files`, y
debería coincidir con lo que mostró la interfaz de Spark para la consulta
de ese año. Esa diferencia de ficheros —no el tiempo que ha tardado la
celda— es la evidencia que vamos a citar en la tabla comparativa de más
abajo. El tiempo lo medimos justo después, como complemento.

### El tiempo de cada consulta

Los ficheros y los bytes leídos son la evidencia principal de la poda, pero
también podemos medir cuánto tarda cada consulta. Para que la comparación
sea justa, las cinco consultas hacen el mismo trabajo, una suma sobre una
columna, y sólo cambian la tabla y el filtro. Cada una se ejecuta una vez
para calentar (cargar clases, planificar) y se anota la mediana de cinco
ejecuciones más. En SQL:

```sql
SELECT SUM(ws_net_profit) FROM web_sales_silver;                                   -- lectura completa
SELECT SUM(ws_net_profit) FROM web_sales_silver WHERE sold_year = <año>;           -- un año
SELECT SUM(ws_net_profit) FROM web_sales_silver WHERE ws_bill_customer_sk = <cliente>;  -- un cliente
-- y las dos últimas, igual, sobre la salida de control sin particionar
```

In [ ]:
from collections.abc import Callable
from statistics import median
from time import perf_counter


def median_seconds(action: Callable[[], object], repetitions: int = 5) -> float:
    """Mediana, en segundos, de varias ejecuciones de action (la primera se descarta)."""
    action()
    durations: list[float] = []
    for _ in range(repetitions):
        started: float = perf_counter()
        action()
        durations.append(perf_counter() - started)
    return median(durations)


def total_profit(table: DataFrame) -> Callable[[], object]:
    return lambda: table.agg(F.sum("ws_net_profit")).collect()


silver_control: DataFrame = spark.read.parquet(silver_control_path)

timings: dict[str, float] = {
    "silver, lectura completa": median_seconds(total_profit(full_scan)),
    f"silver, un año ({sample_year})": median_seconds(total_profit(filtered)),
    f"control, un año ({sample_year})": median_seconds(
        total_profit(silver_control.filter(col("sold_year") == sample_year))
    ),
    f"silver, un cliente ({sample_customer})": median_seconds(total_profit(customer_query)),
    f"control, un cliente ({sample_customer})": median_seconds(
        total_profit(silver_control.filter(col("ws_bill_customer_sk") == sample_customer))
    ),
}
for name, seconds in timings.items():
    print(f"{name:35} {seconds * 1000:7.0f} ms")

No hay `assert` sobre estos tiempos: dependen de la carga del equipo, de las
cachés y de YARN, y con SF3 las diferencias son pequeñas y pueden invertirse
de una ejecución a otra. Lo que esperamos es que la consulta de un año tarde
menos sobre `silver` que sobre la salida de control, porque lee una fracción
de los ficheros, y que la del cliente no mejore con la partición por año y
mes, porque abre todos.

Para ver qué pasaría con más datos, repetimos este mismo experimento fuera
del clúster, con PySpark local y datos de TPC-DS generados con SF3 y con
SF30 (diez veces más). Con SF3, la consulta de un año tardó prácticamente lo
mismo en las dos tablas (91 frente a 102 ms): el coste fijo de
planificar y lanzar la consulta tapa la diferencia de lectura. Con SF30,
la diferencia pasó a ser de 2 veces (116 frente a 234 ms): lo que ahorra la
poda es proporcional a los datos que no se leen, y el coste fijo ya no lo
tapa. Es el argumento a favor de particionar: la ventaja que ahora ves en
ficheros y en bytes crece con el tamaño de la tabla, aunque con este
volumen apenas se note en segundos. Son cifras de una máquina y una
ejecución, no una referencia.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Ejercicio: una segunda tabla de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> por cliente</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>La consulta: historial de compras de un cliente (consulta 5), <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>WHERE ws_bill_customer_sk = &lt;cliente&gt;</code></li><li style='margin:10px 0;list-style-position:outside'>Escribe una copia con sus cuatro columnas ordenada por cliente con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>orderBy</code>, y otra sin ordenar con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>repartition(8)</code></li><li style='margin:10px 0;list-style-position:outside'>Compara <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> y las dos copias: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>PushedFilters</code>, ficheros y bytes leídos, y tiempo con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>median_seconds</code></li><li style='margin:10px 0;list-style-position:outside'>Qué cuesta: espacio en HDFS, y qué habría que añadir al contrato</li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

### Ejercicio: una segunda tabla de `silver` para la consulta del cliente

**La consulta.** Es la consulta 5 de la lista de arriba: el historial de
compras de un cliente —el que tiene más pedidos, `sample_customer`—, con el
número de pedido, la fecha y hora y el importe pagado de cada línea,
ordenado por fecha. Es la que sale perdiendo con la partición por año y mes,
porque `silver` no está organizada por cliente y la consulta abre todos los
ficheros. En SQL:

```sql
SELECT ws_order_number, ws_sold_timestamp, ws_net_paid
FROM web_sales_silver
WHERE ws_bill_customer_sk = <cliente>
ORDER BY ws_sold_timestamp;
```

En Spark, sobre una tabla `table` cualquiera con esas columnas:

```python
table.filter(col("ws_bill_customer_sk") == sample_customer) \
     .select("ws_order_number", "ws_sold_timestamp", "ws_net_paid") \
     .orderBy("ws_sold_timestamp")
```

**Qué hay que hacer.** Para resolverla sin tocar `silver`, construye dos
copias de laboratorio con las mismas cuatro columnas que usa la consulta,
una ordenada por cliente y otra sin ordenar. La segunda sirve para separar
dos efectos: leer cuatro columnas en lugar de once, y tener los datos
ordenados.

1. Parte de `web_sales_silver_read_back`, la lectura de `silver` que ya
   tienes, no de `raw`, y quédate sólo con `ws_bill_customer_sk`,
   `ws_order_number`, `ws_sold_timestamp` y `ws_net_paid`.
2. **Copia ordenada.** Ordénala con `orderBy("ws_bill_customer_sk")`, el
   mismo `orderBy` de S4, y escríbela sin `partitionBy` en
   `/user/luser/s5/web_sales_silver_por_cliente`. Al ordenar, Spark reparte
   las filas por rangos de cliente y las ordena dentro de cada fichero: cada
   fichero reúne un rango de clientes, y el mínimo y el máximo que Parquet
   guarda por grupo de filas dejan de abarcar a casi todos.
3. **Copia sin ordenar.** Escribe las mismas cuatro columnas con
   `repartition(8)`, sin ordenar, en
   `/user/luser/s5/web_sales_cuatro_columnas_sin_ordenar`. Ambas son tablas
   de laboratorio, como la salida de control, y por eso van bajo la ruta de
   trabajo y no bajo `/datalake`.
4. Ejecuta la consulta sobre `silver` y sobre las dos copias, y compara el
   plan (`PushedFilters`), los *files read* y *size of files read* de la
   interfaz de Spark, el tamaño de cada salida en HDFS
   (`hdfs dfs -count -h`) y el tiempo, con `median_seconds` de la sección
   anterior.

Después, responde con tus palabras: ¿qué ha mejorado y qué sigue igual? ¿El
número de ficheros leídos baja, o lo que baja es otra cosa? ¿Qué parte de la
mejora se debe a leer menos columnas y qué parte a la ordenación? ¿Qué le
habrías tenido que añadir al contrato de `silver` si esta tabla fuera a
quedarse? ¿Y qué ocurriría con las consultas 1 a 4 si sólo existiera esta
tabla?

In [ ]:
# Tu solución: escribe las dos copias y ejecuta la consulta del cliente sobre las tres tablas

**Tus observaciones:**

#### Qué cabe esperar

En la simulación fuera del clúster (PySpark local, SF3 y SF30), la consulta
del cliente sobre la copia ordenada tardó la mitad que sobre `silver` con SF3
(99 frente a 230 ms) y cuatro veces menos con SF30 (111 frente a 473 ms), y
frente a la copia de cuatro columnas sin ordenar la diferencia fue mayor aún.
El número de ficheros leídos no bajó: se abren todos los de la copia, y lo
que ahorra el lector de Parquet son los grupos de filas cuyo rango de
clientes no incluye al elegido. Parte del ahorro en bytes viene también de
que los datos ordenados se comprimen mejor. Compara estas cifras con las
tuyas: lo importante es el orden de magnitud y cómo crece con el tamaño de
la tabla, no los milisegundos.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Cambia el filtro y compara la poda</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Escribe la consulta 2, noviembre y diciembre de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sample_year</code>: ¿cuántos ficheros lee ahora?</li><li style='margin:10px 0;list-style-position:outside'>Filtra sólo por <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_month = 12</code>: ¿se poda algo? ¿En qué se diferencia de filtrar sólo por año?</li><li style='margin:10px 0;list-style-position:outside'>Repite la consulta del cliente (<code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>WHERE ws_bill_customer_sk = &lt;cliente&gt;</code>) sobre la salida de control sin particionar: ¿cambia algo en su plan o en los ficheros que lee?</li><li style='margin:10px 0;list-style-position:outside'>Filtra por <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_net_profit &gt; 1000</code>: ¿aparece <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>PartitionFilters</code>? ¿Dónde aparece el filtro?</li><li style='margin:10px 0;list-style-position:outside'>Filtra por <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_sold_date</code> en lugar de por año: ¿se poda? ¿Por qué no, si la fecha determina el año?</li><li style='margin:10px 0;list-style-position:outside'>Lee un único directorio <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sold_month=</code> por su ruta, con y sin <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>basePath</code>: ¿qué columnas cambian?</li><li style='margin:10px 0;list-style-position:outside'>Escribe una copia con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>partitionBy(&quot;sold_year&quot;)</code> en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/user/luser/s5</code> y compara ficheros y tamaño medio</li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

No volveremos a usar la `SparkSession`: el resto de la sesión sólo
mira la estructura de directorios con `hdfs dfs`. La cerramos aquí para
liberar los ejecutores y el ApplicationMaster de YARN, además de la memoria
del driver.

In [ ]:
spark.stop()

## Comprobar la estructura de directorios en HDFS

Miramos ahora directamente en HDFS la estructura `columna=valor` que
Spark ha creado, y la comparamos con la de la salida de control sin
particionar y con la del producto `gold`.

In [ ]:
!hdfs dfs -ls {silver_path}

`hdfs dfs -ls` sólo enumera un nivel de directorios. El recuento del árbol
completo es el `hdfs dfs -count` que usamos para medir la poda; con `-h`
escribe el tamaño en unidades legibles. Las cifras son, en orden,
directorios, ficheros (incluido `_SUCCESS`) y tamaño lógico total:

In [ ]:
!hdfs dfs -count -h {silver_path}

Cada entrada `sold_year=<año>` es un directorio, no un fichero de datos.
Con `-R` recorremos el árbol del año elegido antes, y con `-h` vemos el
tamaño de cada fichero: doce subdirectorios mensuales, cada uno con un
único fichero Parquet, el que dejó el `repartition`. Fíjate en los
tamaños: los meses de agosto a diciembre pesan bastante más que los de
enero a julio, la estacionalidad que vimos al diseñar las particiones.

In [ ]:
!hdfs dfs -ls -R -h {silver_path}/sold_year={sample_year}

Y el año 2003, en el que terminan los datos: un solo mes, con un fichero
varias veces más pequeño que el del mes más pequeño de los años
completos:

In [ ]:
!hdfs dfs -ls -R -h {silver_path}/sold_year=2003

Compara esa estructura con la salida de control, que no tiene ningún
subdirectorio `columna=valor`: todos los fragmentos Parquet cuelgan
directamente de la ruta:

In [ ]:
!hdfs dfs -ls {silver_control_path}

Y con el producto `gold`, que al llevar `coalesce(1)` debería mostrar un
único fichero de datos junto al marcador `_SUCCESS`:

In [ ]:
!hdfs dfs -ls {gold_path}

### Cuánto pesa un fichero

Una última medida: el tamaño medio de los ficheros de datos, que sale de
dividir el tamaño lógico total entre el número de ficheros de datos. Para
`silver` son las dos cifras que guardó `hdfs_count` al medir la poda; para
la salida de control, que tiene las mismas filas sin particionar, se lo
preguntamos ahora a HDFS:

In [ ]:
_, control_files, control_total_bytes = hdfs_count(silver_control_path)
control_data_files: int = control_files - 1  # descarta _SUCCESS

MIB: int = 1024 * 1024
print(
    f"silver:  {full_scan_files} ficheros de datos, "
    f"{silver_total_bytes / full_scan_files / MIB:.1f} MiB de media"
)
print(
    f"control: {control_data_files} ficheros de datos, "
    f"{control_total_bytes / control_data_files / MIB:.1f} MiB de media"
)

Compara esas cifras con el tamaño de bloque de este clúster (64 MiB,
`dfs.blocksize`, S1) y con lo que S4 explicó sobre los ficheros pequeños.
En SF3 ningún fichero de `silver` llena ni de lejos un bloque, y
particionar ha multiplicado el número de ficheros respecto a la salida de
control para los mismos datos: **la pareja año-mes es más fina de lo que
este volumen justifica**. Particionar sólo por `sold_year` daría cinco
ficheros de unos 10 MiB y uno diminuto para 2003. La mantenemos por
motivos docentes, porque permite ver la poda en dos niveles.

Con un dataset mil veces mayor la conclusión cambia: cada mes pesaría
cientos de MiB, varios bloques, y la pareja año-mes estaría justificada.
Como decíamos al diseñar las particiones, la granularidad depende de la
escala. ¿Qué elegirías para SF3 y qué para SF1000: particionar sólo por
año, mantener año y mes, o reagrupar más filas por fichero?

## Tabla comparativa: fuente, refinada y *gold*

Con lo observado en las celdas anteriores se pueden comparar las tres
versiones del dato que anunciaba la introducción —más la salida de
control—, con evidencia concreta de esta ejecución en lugar de con texto
genérico:

| Versión | Organización | Filas | Lo que aporta / responde |
| --- | --- | --- | --- |
| Fuente (`raw`) | `/datalake/raw/tpcds/web_sales`, un directorio sin particionar | `raw_count` filas (2.160.165 en SF3) | Qué datos existen físicamente; es la referencia inmutable frente a la que se valida todo lo demás |
| Refinada sin particionar | `/user/luser/s5/web_sales_silver_sin_particionar`, un directorio con pocos ficheros grandes (`control_data_files`), mismas filas que la fuente menos las que no tienen fecha | `silver_count` filas: `raw_count` menos `missing_date_count` (comprobado con `assert`), no `raw_count` exacto | Qué aporta la transformación por sí sola: columnas de negocio limpias, `ws_sold_date` derivada, líneas sin fecha ya descartadas de forma documentada, sin reorganización física todavía |
| Refinada particionada | `/datalake/silver/tpcds/web_sales/sold_year=.../sold_month=...`, un subdirectorio por combinación año/mes, con un fichero pequeño en cada uno (`full_scan_files`) | Las mismas filas que la versión sin particionar, repartidas entre directorios (ver `year_counts` más arriba) | Qué datos puede evitar leer una consulta que se limita a un año: compara `full_scan_files` con `filtered_files` — si el segundo es menor, la poda de particiones ha descartado directorios completos. La consulta de un cliente, en cambio, abre todos los ficheros |
| Gold | `/datalake/gold/tpcds/web_sales_by_year`, un único fichero | Una fila por año (`sold_year`) | Qué producto consume directamente una pregunta de negocio (pedidos, líneas y beneficio neto por año), sin que quien lo consulte tenga que repetir la agregación |

Las cifras de esta tabla no son números fijos de esta plantilla: son los
valores impresos por `raw_count`, `silver_count`,
`missing_date_count`, `year_counts`, `full_scan_files`,
`filtered_files` y `control_data_files` en las celdas de arriba, en esta misma ejecución del
notebook. Vuelve a mirarlos si repites la sesión más adelante.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Los mismos datos, cuatro organizaciones</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>La poda se ve dos veces: en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>PartitionFilters</code> del plan y en menos ficheros contados en HDFS (no en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>inputFiles()</code>, que no la refleja)</li><li style='margin:10px 0;list-style-position:outside'>Las cifras de la tabla comparativa son las de esta ejecución: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw_count</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver_count</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>full_scan_files</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>filtered_files</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>control_data_files</code></li><li style='margin:10px 0;list-style-position:outside'>Un directorio <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>columna=valor</code> bien organizado sigue sin ser una tabla gestionada: sin catálogo, nombre, historial ni transacciones</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Mañana quieres leer <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> desde otro motor: ¿qué tienes que saber de memoria?</li><li style='margin:10px 0;list-style-position:outside'>Si quien escribió este notebook se marcha, ¿cómo se entera el siguiente de dónde está la tabla y qué significa cada columna?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Lo que un directorio bien organizado no resuelve</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>El nombre de la tabla no está centralizado: cada motor necesita conocer la ruta física</li><li style='margin:10px 0;list-style-position:outside'>Una partición nueva no queda registrada en ningún catálogo</li><li style='margin:10px 0;list-style-position:outside'>No hay historial, ni commit atómico, ni transacciones para actualizar o borrar filas</li><li style='margin:10px 0;list-style-position:outside'>Tampoco una política común de metadatos y estadísticas</li></ul><blockquote style='background:#ffffff;color:inherit;border:none;border-left:6px solid #174f7a;margin:22px 0 0;padding:12px 16px'><div style='margin-top:0px'>Son los problemas que resolverán Hive Metastore (S6) e Iceberg (S7).</div></blockquote><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Buenas prácticas que quedan pendientes del catálogo

Aunque `/datalake/silver/tpcds/web_sales` y
`/datalake/gold/tpcds/web_sales_by_year` ya siguen convenciones razonables
de organización, siguen siendo colecciones de Parquet sobre HDFS, no tablas
gestionadas. Eso deja límites concretos que S5 no resuelve:

- el nombre lógico de la tabla (`web_sales`, en `silver`) no está
  centralizado en ningún sitio: no hay un servicio al que preguntar
  "¿dónde está la tabla `web_sales` de `silver`?", así que cada motor
  que quiera leerla necesita conocer de antemano la ruta física exacta,
  como hemos hecho aquí a mano;
- una partición nueva —por ejemplo, si llegaran ventas de un año
  posterior— puede escribirse en el directorio correcto y, aun así, no
  estar "registrada" para un cliente que dependa de una lista de
  particiones cacheada o de un catálogo. En S6, cuando la tabla esté
  registrada en el Hive Metastore, habrá que avisarle de cada partición
  nueva con `MSCK REPAIR TABLE` (o `sync_partition_metadata` desde
  Trino). Quien descubre la tabla por rutas, como hace este mismo
  notebook al leer desde la raíz, sí vería la partición nueva de
  inmediato;
- no hay un historial de cambios de la tabla: si alguien vuelve a ejecutar
  esta sesión con otra lógica de transformación, la versión anterior
  simplemente desaparece con el `overwrite`, sin dejar rastro de qué había
  antes;
- no existe un commit atómico de todos los ficheros de una escritura: si
  el proceso se interrumpiera a mitad de la escritura particionada, un
  lector podría ver una tabla a medio escribir;
- las actualizaciones y eliminaciones de filas concretas requerirían
  coordinación manual entre quien escribe y quien lee, porque no hay
  transacciones;
- los metadatos y las estadísticas (número de filas por partición, rangos
  de columnas...) no tienen todavía una política común: cada motor que lea
  esta tabla tendría que recalcularlas por su cuenta.

Estas limitaciones no hacen que el data lake construido en esta sesión
sea incorrecto: son exactamente los problemas que resolverán el Hive
Metastore en S6 y el formato de tabla Iceberg en S7, y conviene haberlos
sentido de primera mano —intentando, por ejemplo, acordarse a mano de la
ruta física de `web_sales`— antes de ver cómo un catálogo los resuelve.

## Relación con HDFS y S3

La organización `columna=valor` que hemos aplicado en esta sesión puede
utilizarse tanto en HDFS como en un almacenamiento de objetos compatible
con S3, como el que se introdujo en la sesión 3. Sin embargo, un
almacenamiento de objetos no tiene directorios reales: lo que aquí llamamos "directorio" es, en un backend
S3, una convención sobre el prefijo de las claves de objeto. Conviene
mantener separadas tres ideas que es fácil mezclar:

- la semántica del formato Parquet y del particionado por directorios, que
  es la misma en ambos casos;
- las operaciones concretas de HDFS (`hdfs dfs -ls`, bloques, el NameNode)
  o de la API S3 (listar por prefijo, objetos), que son mecanismos
  distintos para conseguir el mismo efecto observable;
- la gestión de catálogo que se incorporará después, en S6 y S7, y que es
  independiente del almacenamiento subyacente.

Hay una diferencia de mecanismo que sí se nota al escribir. Spark
escribe primero en un directorio temporal y confirma la escritura
renombrándolo, algo que en HDFS es una operación barata del NameNode. S3
no tiene renombrado: "mover" un objeto es copiarlo y borrar el
original, uno a uno. Por eso, en un almacenamiento de objetos, el
problema del commit atómico de la sección anterior es todavía más
serio, y es uno de los motivos por los que existen formatos de tabla
como Iceberg (S7), que confirman una escritura sin renombrar ficheros.

La sesión 3 ya mostró que el mismo Parquet puede copiarse a un almacenamiento
S3-compatible sin modificar su esquema. S5 continúa ese recorrido
centrándose en la organización física y el particionado de los ficheros,
no en dónde viven los bytes.

## Preguntas para interpretar la experiencia

- ¿Por qué conservar `/datalake/raw/tpcds` sin modificar facilita comparar
  esta sesión con las anteriores, incluso si cambiamos por completo la
  lógica de la transformación `silver`?
- Da un ejemplo, con tus propias palabras, de la diferencia entre una
  partición de ejecución de Spark y una partición Hive como las que hemos
  creado aquí. ¿Podrías tener muchas particiones de ejecución sobre una
  tabla con una única partición Hive, o al revés?
- ¿Por qué `ws_bill_customer_sk` sería una mala columna de partición
  principal para `web_sales`, aunque muchas consultas de negocio se
  limiten a un cliente, como la consulta 5? ¿Qué mostró su plan?
- En la comparación de planes, ¿qué diferencia concreta viste entre el plan
  de la lectura completa y el de la consulta de un año? ¿Coincidió con la
  diferencia de ficheros contados en HDFS? ¿Por qué `inputFiles()` no sirve
  para medir esto?
- ¿Por qué el enunciado insiste en que el tiempo de ejecución de una
  celda en un portátil no demuestra, por sí solo, que el particionado
  ayuda? ¿Qué evidencia usaste en su lugar?
- La salida de control sin particionar vive en `/user/luser/s5`, no en
  `/datalake/silver`. ¿Por qué tiene sentido esa distinción de rutas para
  un artefacto que sólo sirve para esta comparación?
- Explica qué comprueban los `assert`
  de esta sesión y por qué cada uno detecta un problema distinto (filas
  perdidas frente a fechas o claves nulas). ¿Por qué unas claves deben
  ser siempre no nulas y de otras basta con saber cuántas faltan?
- A la vista del tamaño medio de los ficheros de `silver` y de la salida
  de control, ¿qué columnas de partición elegirías para SF3 y cuáles para
  SF1000? ¿Por qué?
- De la lista de límites pendientes del catálogo, elige uno y describe con
  un ejemplo concreto de esta sesión en qué momento lo has notado.

## Evidencias para la siguiente sesión

Antes de la siguiente sesión tendrás una reunión individual breve con el
profesor para revisar el trabajo de esta sesión. Esa reunión combina una
demostración en vivo sobre tu propio ordenador y una memoria escrita breve.

### Qué mostrar en el ordenador durante la reunión

Ten preparadas estas evidencias:

1. La ruta fuente inmutable (`/datalake/raw/tpcds/web_sales`) y su número
   de filas, sin modificar.
2. La ruta refinada `/datalake/silver/tpcds/web_sales`, con su esquema
   (`printSchema`), la comprobación frente al esquema declarado
   (`web_sales_silver_schema`) y la definición de sus particiones
   (`sold_year`, `sold_month`).
3. El producto `gold` `/datalake/gold/tpcds/web_sales_by_year`, derivado de
   `silver`, con su contenido (`show()`) y su esquema releído, que coincide
   con `web_sales_by_year_schema`.
4. La comprobación de filas y claves: el resultado de los `assert` de esta
   sesión, junto con los valores que compararon (`raw_count`,
   `silver_count`, `missing_date_count`, `null_dates`, `null_keys`).
5. El listado `hdfs dfs -ls -R -h` de un año de la tabla particionada,
   con los tamaños de sus ficheros, y la comparación del tamaño medio de
   los ficheros de `silver` con el de la salida de control.
6. La comparación entre la lectura completa y la consulta de un año: los
   dos planes de `.explain("formatted")` (con sus `PartitionFilters`), el
   plan de la consulta de un cliente (sin ellos), los
   ficheros leídos por cada consulta según la interfaz de Spark y los dos
   recuentos de ficheros en HDFS (`full_scan_files`, `filtered_files`).
7. Una explicación, con tus propias palabras, de por qué `sold_year`/
   `sold_month` es una partición útil para `web_sales` y por qué
   `ws_order_number`, `ws_item_sk` o `ws_bill_customer_sk` no lo serían.

Las evidencias que salen de Spark (esquemas, planes y recuentos) necesitan
una `SparkSession` viva: como el notebook la cierra con `spark.stop()`, para
enseñarlas en la reunión hay que volver a ejecutar las celdas desde «Crear
una SparkSession sobre YARN».

### Memoria escrita (una o dos páginas)

Trae también un documento breve —una o dos páginas, no hace falta más— que
no se limite a pegar capturas de las salidas anteriores: debe explicar con
tus propias palabras qué garantiza cada zona del data lake (`raw`, `silver`
y `gold`); qué dice el contrato de las dos salidas de esta sesión; por qué
`sold_year` y `sold_month` son una buena partición y una clave de cliente,
de artículo o de pedido no lo sería; qué evidencia demuestra que la
consulta de un año evita leer ficheros y por qué la de un cliente no; si
la granularidad año-mes está justificada para SF3; y qué sigue sin
resolver una colección de Parquet bien organizada pero sin catálogo.

### Qué es importante de cara al examen final

El examen no pide recordar la sintaxis exacta de una función. Debes poder
explicar:

- la arquitectura medallion y qué se espera de los datos en cada zona;
- qué debe decir el contrato de una salida de datos: ruta, columnas y sus
  tipos, claves de unión, significado de cada métrica, formato y regla de
  regeneración;
- qué comprueba un esquema declarado al escribir y qué hace Spark al leer
  con él, y por qué no sirve para garantizar que una columna no tiene
  nulos;
- qué es el particionado físico por directorios `columna=valor` y en qué se
  distingue de una partición de ejecución de Spark, de un bloque HDFS y de
  una transformación de partición de Iceberg;
- con qué criterios se elige una columna de partición: las consultas
  habituales, la cardinalidad y el tamaño de los ficheros resultantes, y
  por qué la granularidad adecuada depende de la escala;
- qué comprobaciones de calidad conviene hacer sobre una salida (filas
  perdidas frente a la fuente, claves y columnas derivadas nulas, rangos)
  y qué detecta cada una, y qué añade, si la salida cruza tablas,
  comprobar que el `JOIN` no pierde ni multiplica filas;
- cómo se reconoce la poda de particiones en un plan y por qué el tiempo de
  una celda no es una evidencia fiable;
- qué limitaciones tiene un data lake de ficheros sin catálogo.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Sesión 5</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Medallion aplicado: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code> intacto, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> validado y particionado, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code> orientado a una pregunta de negocio</li><li style='margin:10px 0;list-style-position:outside'>Las particiones Hive son directorios <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>columna=valor</code>: ni particiones de ejecución, ni bloques HDFS, ni transformaciones Iceberg</li><li style='margin:10px 0;list-style-position:outside'>Sin catálogo: cada motor conoce la ruta física y la descubre a mano</li></ul><blockquote style='background:#ffffff;color:inherit;border:none;border-left:6px solid #a54467;margin:22px 0 0;padding:12px 16px'><div style='margin-top:0px'>Siguiente bloque: el Hive Metastore registra esta organización como tabla Hive (S6)</div></blockquote><hr color="#a54467" size="2" noshade style="display:none"></div>

## Detener el laboratorio

Después de guardar el notebook, ejecuta `make -C entorno stop` en una
terminal de tu equipo, desde la raíz de la distribución. Se detienen los
servicios Compose incluidos en `entorno/` (también RustFS) y Jupyter, pero
HDFS conserva `raw`, `silver` y `gold` para S6. Para continuar, ejecuta
`make -C entorno hadoop-up` y vuelve a iniciar Jupyter como en «Antes de
empezar»; no ejecutes `clean-hdfs` mientras avances por el curso.


## Siguiente paso

S5 deja preparados los tres niveles del recorrido medallion: una fuente
inmutable, una tabla `silver` particionada físicamente y un producto `gold`
derivado de ella. S6 registrará las rutas `silver` y `gold` como tablas
externas Hive, mostrará las particiones desde un metastore y permitirá que
Spark y Trino consulten la misma definición de tabla sin que cada motor
tenga que conocer de antemano la ruta física ni volver a listar los
directorios por su cuenta.